In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF set.")

PYTORCH_CUDA_ALLOC_CONF set.


In [2]:
import sys
import subprocess

packages = [
    "langchain==1.3.9",
    "langchain-core==1.4.7",
    "langchain-community==0.4.2",
    "langchain-huggingface==1.2.2",
    "langchain-chroma==1.1.0",
    "langchain-text-splitters==1.1.2",
    "sentence-transformers==3.0.1",
    "chromadb",
    "pymupdf",
    "pdfplumber",
    "rank-bm25",
    "bitsandbytes",
    "accelerate",
    "scikit-learn",
    "bert-score",
]

subprocess.run(["pip", "install", "-q", "--no-cache-dir"] + packages, check=True)

#Auto-restart kernel so all installs are visible immediately

#print("✅ Packages installed — restarting kernel...")

#import IPython

#IPython.Application.instance().kernel.do_shutdown(restart=True)

CompletedProcess(args=['pip', 'install', '-q', '--no-cache-dir', 'langchain==1.3.9', 'langchain-core==1.4.7', 'langchain-community==0.4.2', 'langchain-huggingface==1.2.2', 'langchain-chroma==1.1.0', 'langchain-text-splitters==1.1.2', 'sentence-transformers==3.0.1', 'chromadb', 'pymupdf', 'pdfplumber', 'rank-bm25', 'bitsandbytes', 'accelerate', 'scikit-learn', 'bert-score'], returncode=0)

In [3]:
from pathlib import Path
from langchain_core.documents import Document
import fitz        
import pdfplumber
import numpy as np  
pdf_dir = "/kaggle/input/datasets/shivammusk/sec-filings/SEC Filings"
pdf_files = list(Path(pdf_dir).glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files\n")

all_documents = []

for pdf_path in pdf_files:
    print(f"Processing: {pdf_path.name}")
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text").strip()

        if len(text) < 30:
            continue

        # Text block
        all_documents.append(Document(
            page_content=text,
            metadata={
                "source": str(pdf_path),
                "file_name": pdf_path.name,
                "element_type": "Text",
                "page_number": page_num + 1,
            }
        ))

        # Table extraction
        try:
            with pdfplumber.open(pdf_path) as pdf:
                plumber_page = pdf.pages[page_num]
                tables = plumber_page.extract_tables()
                for idx, table in enumerate(tables):
                    if table and len(table) > 1:
                        table_text = "\n".join(
                            [" | ".join(str(cell) if cell is not None else "" for cell in row)
                             for row in table]
                        )
                        all_documents.append(Document(
                            page_content=table_text,
                            metadata={
                                "source": str(pdf_path),
                                "file_name": pdf_path.name,
                                "element_type": "Table",
                                "page_number": page_num + 1,
                                "table_index": idx,
                            }
                        ))
        except Exception:
            continue

    doc.close()

print(f"\n✅ Extraction complete!")
print(f"Total Documents : {len(all_documents)}")
print(f"Text Blocks     : {sum(1 for d in all_documents if d.metadata['element_type'] == 'Text')}")
print(f"Tables          : {sum(1 for d in all_documents if d.metadata['element_type'] == 'Table')}")


Found 5 PDF files

Processing: Oracle.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Processing: Meta.pdf
Processing: Tesla.pdf
Processing: Nvidia.pdf
Processing: Apple.pdf

✅ Extraction complete!
Total Documents : 1043
Text Blocks     : 755
Tables          : 288


In [4]:
import yaml

def to_okf_concept(doc):
    company_guess = doc.metadata.get("file_name", "Unknown").split(".")[0].replace("_", " ")
    frontmatter = {
        "type": "FinancialTable" if doc.metadata.get("element_type") == "Table" else "FinancialText",
        "company": company_guess,
        "source_file": doc.metadata.get("file_name", "Unknown"),
        "page": doc.metadata.get("page_number", "?"),
    }
    fm_str = yaml.dump(frontmatter, sort_keys=False)
    doc.page_content = f"---\n{fm_str}---\n\n{doc.page_content.strip()}"
    return doc

all_documents = [to_okf_concept(d) for d in all_documents]
print(f"✅ Wrapped {len(all_documents)} documents as OKF concept blocks (frontmatter + content)")


✅ Wrapped 1043 documents as OKF concept blocks (frontmatter + content)


In [5]:
import subprocess
import sys

# 1. Wipe ALL related cached modules
to_remove = [k for k in sys.modules if any(x in k for x in 
    ["sentence", "langchain_huggingface", "huggingface", "langchain_core", "langchain"])]
for mod in to_remove:
    del sys.modules[mod]

# 2. Reinstall both together
subprocess.run(["pip", "install", "-q", "--no-cache-dir",
    "sentence-transformers==3.0.1",
    "langchain-huggingface==1.2.2"], check=True)

# 3. Verify sentence_transformers loads directly first
import sentence_transformers
print("sentence_transformers version:", sentence_transformers.__version__)

# 4. Now load HuggingFaceEmbeddings fresh
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)
print("✅ Embedding model loaded!")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-09-01 04:40:07.076559: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788237607.279036     146 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788237607.340889     146 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788237607.850351     146 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the sam

sentence_transformers version: 3.0.1


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!


In [6]:
pip install langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 11.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [7]:
# CELL 6: Faster Semantic Chunking (GPU Optimized)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
import torch
import gc

# Clear memory
torch.cuda.empty_cache()
gc.collect()

print(f"Current GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Use GPU with small batch size
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 2          # Small batch = less memory
    }
)

TARGET_COMPANIES = ["tesla","nvidia"]

def _is_target_company(doc):
    fname = doc.metadata.get("file_name","").lower()
    return any(c in fname for c in TARGET_COMPANIES)

target_docs = [doc for doc in all_documents if _is_target_company(doc)]
skipped_docs = [doc for doc in all_documents if not _is_target_company(doc)]

table_docs = [doc for doc in target_docs if doc.metadata.get("element_type") == "Table"]
text_docs = [doc for doc in target_docs if doc.metadata.get("element_type") == "Text"]

print(f"Target docs (Tesla + Nvidia): {len(target_docs)}  |  Skipped (other companies): {len(skipped_docs)}")
print(f"  -> Tables: {len(table_docs)} | Text: {len(text_docs)}")

semantic_splitter = SemanticChunker(
    embeddings = embeddings,
    breakpoint_threshold_type = "percentile",
    breakpoint_threshold_amount = 90,
)

print("🔄 Performing semantic chunking on GPU (Tesla & Nvidia only)...")
text_chunks = semantic_splitter.split_documents(text_docs)

chunks = table_docs + text_chunks

del embeddings, semantic_splitter
torch.cuda.empty_cache()
gc.collect()

print(f"✅ Done!")
print(f"Tables: {len(table_docs)} | Text Chunks: {len(text_chunks)} | Total: {len(chunks)}")
print(f"GPU memory now: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"ℹ️  Note: {len(skipped_docs)} documents from other companies (Oracle, Meta, Apple) were excluded from chunking/vectorstore.")



/tmp/ipykernel_146/3161353956.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Current GPU memory: 0.41 GB


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Target docs (Tesla + Nvidia): 390  |  Skipped (other companies): 653
  -> Tables: 118 | Text: 272
🔄 Performing semantic chunking on GPU (Tesla & Nvidia only)...
✅ Done!
Tables: 118 | Text Chunks: 701 | Total: 819
GPU memory now: 0.42 GB
ℹ️  Note: 653 documents from other companies (Oracle, Meta, Apple) were excluded from chunking/vectorstore.


In [8]:
sample_embedding = embedding_model.embed_query(
    chunks[0].page_content
)

print("Embedding dimension:", len(sample_embedding))

Embedding dimension: 768


In [9]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory = "./financial_db" 
)

print("✅ Vector Store Created and Persisted")

✅ Vector Store Created and Persisted


In [10]:
query = "What is NVIDIA's total revenue?"
results = vectorstore.similarity_search(query,k = 3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {doc.metadata['file_name']} | Page: {doc.metadata.get('page_number')}")
    print(f"Type: {doc.metadata['element_type']}")
    print(doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content)


--- Result 1 ---
Source: Nvidia.pdf | Page: 94
Type: Text
---
type: FinancialText
company: Nvidia
source_file: Nvidia.pdf
page: 94
---

Table of Contents
NVIDIA Corporation and Subsidiaries
Notes to the Consolidated Financial Statements
(Continued)
We recognized revenue of $974 million and $729 million in fiscal years 2026 and 2025, respectively, that were included in the prior year
end deferred revenue balance. As of January 25, 2026, revenue related to remaining performance obligations from contracts greater than one year in length was $2.3
billion, ...

--- Result 2 ---
Source: Nvidia.pdf | Page: 57
Type: Table
---
type: FinancialTable
company: Nvidia
source_file: Nvidia.pdf
page: 57
---

Revenue | 100.0 | % |  | 100.0 | %
Cost of revenue | 28.9 |  |  | 25.0 | 
Gross profit | 71.1 |  |  | 75.0 | 
Operating expenses |  |  |  |  | 
Research and development | 8.6 |  |  | 9.9 | 
Sales, general and administrative | 2.1 |  |  | 2.7 | 
Total operating expenses | 10.7 |  |  | 12.6 | 
Opera

In [11]:
!pip install -q langchain langchain-community

In [12]:
import os
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ── ChatMessageHistory: moved to langchain-core in 1.x
from langchain_core.chat_history import InMemoryChatMessageHistory
from types import SimpleNamespace
from langchain_core.tools import tool

# Cross Encoder for reranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Per-session memory store
_memory_store = {}

def get_memory(company=None, session_id="default"):
    key = (company.lower().strip() if company else None, session_id)
    if key not in _memory_store:
        _memory_store[key] = InMemoryChatMessageHistory()
    return _memory_store[key]

def _strip_sec_header(text: str) -> str:
    if "[SEC FILING DATA]" not in text:
        return text.strip()
    parts = text.split("---\n", maxsplit=2)
    return parts[2].strip() if len(parts) >= 3 else text.strip()

def _clean_text(text):
    markers = ["<think>", "</think>", "**Final Answer**", "Final Answer:", "Changes made:"]
    for m in markers:
        if m in text:
            text = text.split(m)[0]
    return re.sub(r'\n+(I have|Note that|Please note).*', '', text,
                  flags=re.IGNORECASE | re.DOTALL).strip()

# ============================================================
# Calculator Tool
# ============================================================
@tool
def calculator(expression: str) -> str:
    """
    Use this tool for ANY mathematical calculation in financial analysis.
    Especially for percentage changes, differences, ratios, margins, and growth rates.
    
    Examples of good inputs:
    - ((215938 - 130497) / 130497) * 100
    - 193737 - 115186
    - (153463 / 215938) * 100
    """
    try:
        expr = expression.replace(",", "").replace("$", "").strip()
        expr = expr.replace("%", "/100")
        
        # Safety: only allow math characters
        if not re.match(r'^[\d\s\+\-\*\/\(\)\.]+$', expr):
            return "Error: Invalid characters in expression"
        
        result = eval(expr)
        return f"{result:.4f}" if isinstance(result, float) else str(result)
    except Exception as e:
        return f"Calculation Error: {str(e)}"

print("✅ Core utilities + Calculator tool loaded")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Core utilities + Calculator tool loaded


In [ ]:
import subprocess, sys
subprocess.run(["pip", "install", "-q", "langchain-groq"], check=True)

import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "API_KEY"

chat_llm = ChatGroq(
    model = "qwen/qwen3.8-27b",
    temperature = 0.4,
    max_tokens = 2500,
    model_kwargs = {"top_p": 0.90},
)

llm = chat_llm

print("✅ Using qwen/qwen3.8-27b")
print("`llm`      -> used by financial_rag()")
print("`chat_llm` -> used by the tool-calling agent")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.9 MB/s eta 0:00:00
✅ Using qwen/qwen3.8-27b
`llm`      -> used by financial_rag()
`chat_llm` -> used by the tool-calling agent


In [14]:
tools = [calculator]
chat_llm_with_tools = chat_llm.bind_tools(tools)

print("✅ Calculator tool bound to chat_llm_with_tools")

✅ Calculator tool bound to chat_llm_with_tools


In [15]:
# Hybrid Retrieval (Semantic + BM25)
def hybrid_retrieval(query, vectorstore, company=None, k=50):
    # 1. Semantic search
    results = vectorstore.similarity_search_with_score(query, k=k * 3 if company else k)

    semantic_list = []
    for doc, score in results:
        if company and company.lower() not in doc.metadata.get("source", "").lower():
            continue
        semantic_list.append((doc, 1.0 / (1.0 + score)))

    # 2. BM25 keyword search
    all_data = vectorstore._collection.get(include=["documents", "metadatas"])
    filtered_texts, filtered_metas = [], []

    for text, meta in zip(all_data["documents"], all_data["metadatas"]):
        if company and company.lower() not in str(meta.get("source", "")).lower():
            continue
        filtered_texts.append(_strip_sec_header(text))
        filtered_metas.append(meta)

    bm25_list = []
    if filtered_texts:
        bm25 = BM25Okapi([t.lower().split() for t in filtered_texts])
        scores = bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[::-1][:k]
        for i in top_idx:
            if scores[i] > 0:
                score_norm = scores[i] / max(scores.max(), 1)
                doc = SimpleNamespace(page_content=filtered_texts[i], metadata=filtered_metas[i])
                bm25_list.append((doc, score_norm))

    # Merge semantic + BM25
    merged = {id(d[0]): d for d in semantic_list}
    for doc, score in bm25_list:
        merged[id(doc)] = (doc, merged.get(id(doc), (None, 0))[1] + score * 0.7)

    return sorted(merged.values(), key=lambda x: x[1], reverse=True)[:k]


In [16]:
#  Reranking, multimodal boost, corrective RAG
def rerank_with_cross_encoder(query, candidates, top_n=15):
    if not candidates:
        return []
    docs  = [pair[0] for pair in candidates]
    texts = [_strip_sec_header(d.page_content) for d in docs]
    scores = cross_encoder.predict([[query, t] for t in texts])
    return sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)[:top_n]


def multimodal_boost(reranked_pairs):
    boosted = []
    for doc, score in reranked_pairs:
        new_score = float(score)
        et = doc.metadata.get("element_type", "")
        text = doc.page_content.lower()

        # Strong boost for real tables
        if et == "Table":
            new_score += 0.45

        # Extra boost for income-statement / segment pages
        keywords = [
            "year ended", "consolidated statements of income",
            "revenue by", "$ in millions", "gross profit",
            "operating income","operating expense", "net income"
        ]
        
        if any(k in text for k in keywords):
            new_score += 0.25

        boosted.append((doc, new_score))
    return sorted(boosted, key=lambda x: x[1], reverse=True)


def evaluate_retrieval_quality(query, docs):
    if not docs or len(docs) < 3:
        return False
    combined_text = " ".join(
        _strip_sec_header(doc.page_content)[:1000] for doc, _ in docs[:4]
    ).lower()
    query_words = [w for w in query.lower().split() if len(w) > 3]
    if not query_words:
        return True
    overlap = sum(1 for w in query_words if w in combined_text)
    required_overlap = max(2, len(query_words) // 3)
    print(f"   Retrieval Quality: {overlap}/{required_overlap} words matched")
    return overlap >= required_overlap

print("✅ Reranking + CRAG utilities loaded")


✅ Reranking + CRAG utilities loaded


In [17]:
# Conversation memory helpers
def _build_history_text(memory, max_turns=3):
    msgs  = memory.messages
    pairs = []
    i = 0
    while i < len(msgs) - 1:
        if msgs[i].type == "human" and msgs[i + 1].type == "ai":
            pairs.append((msgs[i].content, msgs[i + 1].content))
            i += 2
        else:
            i += 1
    recent = pairs[-max_turns:]
    if not recent:
        return ""
    lines = ["Previous conversation:"]
    for turn_idx, (q, a) in enumerate(reversed(recent), 1):
        short_a = a[:500] + "…" if len(a) > 500 else a
        lines.append(f"\n[Turn {turn_idx}] User: {q}")
        lines.append(f"AI: {short_a}")
    return "\n".join(lines)

print("✅ Memory helpers loaded")


✅ Memory helpers loaded


In [18]:
# Main financial_rag() function 
def financial_rag(query: str, company: str = None, session_id: str = "default"):
    global vectorstore, embedding_model, llm

    if not all([vectorstore, embedding_model, llm]):
        return "❌ Error: vectorstore, embedding_model or llm not initialized."

    memory = get_memory(company, session_id)

    # Truncate query for clean logging (no prompt leak)
    query_preview = (query.strip()[:60] + "...") if len(query.strip()) > 60 else query.strip()
    print(f"🔍 Company: {company or 'All'} | Query: {query_preview}")

    # ── Retrieval ──────────────────────────────────────────────────────────
    candidates = hybrid_retrieval(query, vectorstore, company=company, k=40)
    reranked   = rerank_with_cross_encoder(query, candidates, top_n= 7)
    reranked   = multimodal_boost(reranked)

    # ── Corrective RAG ─────────────────────────────────────────────────────
    if not evaluate_retrieval_quality(query, reranked):
        print("⚠️  Corrective RAG triggered — widening search...")
        candidates = hybrid_retrieval(query, vectorstore, company=company, k=50)
        reranked   = rerank_with_cross_encoder(query, candidates, top_n=9)
        reranked   = multimodal_boost(reranked)

    # ── Filter & cap ───────────────────────────────────────────────────────
    filtered_docs = [
        (doc, score) for doc, score in reranked
        if not company or company.lower() in str(doc.metadata.get("source", "")).lower()
    ][:16]

    if len(filtered_docs) < 3:
        return f"❌ Not enough relevant information found for '{company}'."

    context = "\n\n---\n\n".join(_strip_sec_header(doc.page_content) for doc, _ in filtered_docs)

    # ── Pass 1: Generate Response ───────────────────────────────────────────
    print("📝 Pass 1: Generating Response")
    pass1_prompt = f"""You are a Senior Institutional Financial Analyst.

ABSOLUTE RULES — NEVER BREAK THEM:

1. Use ONLY numbers that appear VERBATIM in the Context below.
2. Use the context provided below
3. Never invent, estimate, round, or pull any number from memory or training data.
4. LABEL LOCKING (critical):
   - Every number must stay attached to the exact same label it has in the Context.
   - Operating Expenses numbers can ONLY be used for Operating Expenses.
   - Operating Income numbers can ONLY be used for Operating Income.
   - Revenue numbers can ONLY be used for Revenue.
   - Gross Profit / Gross Margin numbers can ONLY be used for Gross Profit / Gross Margin.
   - Net Income numbers can ONLY be used for Net Income.
   - Research & Development, SG&A, and other line items must also keep their own numbers.
   - Never swap or mix numbers between different metrics.

5. When you write a number, always pair it with its full correct label, for example:
   “Operating expenses were $23,076 million”
   “Operating income was $130,387 million”
   Never write a bare number and later assign it to a different metric.

6. DIRECTION RULE:
   - Before writing “increased”, “decreased”, “rose”, “declined”, “growth”, or “drop”, 
     first compare the two numbers belonging to the SAME metric.
   - Later > Earlier → must say “increased” or “rose”.
   - Later < Earlier → must say “decreased” or “declined”.
   - Never assume direction from the movement of expenses or from surrounding text.

7. If either year is missing for a metric, write exactly: “percentage change not available in the retrieved sections.”

9. If a metric is not present in the Context, write: “not disclosed in the retrieved sections.”

11. Write in clear Finincail tone

FIRST, decide the type of question:

A. If the question is mainly about financial figures, trends, revenue, margins, expenses, income, cash flow, etc.:
   → Present the key figures in a clean Markdown table with columns such as:
     | Metric                  | Earlier Year | Later Year | Change | % Change          |
     |-------------------------|--------------|------------|--------|-------------------|
   → After the table, write a structured analysis using only the numbers from the table.
   → For every % Change, show the calculation (example: ((130387-81453)/81453 × 100 = 60.1%)).

B. If the question is about architecture, technology, products (Blackwell, Rubin, etc.), strategy, risks, competition, outlook, or any non-numeric topic:
   → Do NOT create a financial table.
   → Directly write a clear, structured analysis based on the Context.
   → Only mention numbers if they are relevant and present in the Context.

**Context:**
{context}

**Question:**
{query}

**Output Format:**
Provide a structured, concise, and accurate financial analysis.

Financial Analysis:"""

    raw_pass1 = llm.invoke(pass1_prompt)
    final_response = raw_pass1.content if hasattr(raw_pass1, "content") else str(raw_pass1)
    final_response = _clean_text(final_response)


   
    # ── Memory ─────────────────────────────────────────────────────────────
    memory.add_user_message(query)
    memory.add_ai_message(final_response)

    # ── Sources ────────────────────────────────────────────────────────────
    sources = [
        f"{doc.metadata.get('file_name', 'Unknown')} | Page {doc.metadata.get('page_number', '?')}"
        for doc, _ in filtered_docs
    ]
    unique_sources = list(dict.fromkeys(sources))

    final_output = (
        f"# Financial Analysis — {company or 'All Companies'}\n\n"
        + final_response
        + "\n\n## Sources\n"
        + "\n".join(f"- {s}" for s in unique_sources)
    )

    # Debug metadata
    financial_rag._last_context = context
    financial_rag._last_sources = unique_sources
    financial_rag._debug = {
        "initial_docs": len(candidates),
        "final_docs": len(filtered_docs),
        "corrective_triggered": not evaluate_retrieval_quality(query, reranked),
    }

    return final_output

print("✅ financial_rag() is ready")


✅ financial_rag() is ready


In [19]:
# CELL 17b: BERTScore — measures how well the generated answer is grounded in retrieved context 
from bert_score import score as bert_score

_bert_log = []  # stores {query, company, precision, recall, f1} for every call

def compute_bertscore(reference_text: str, candidate_text: str):
    """
    BERTScore of candidate_text against reference_text.
    Here reference = retrieved source context, candidate = generated answer.

    Unlike BLEU, this compares contextual embeddings of tokens instead of
    exact word matches, so a well-paraphrased but accurate answer still
    scores high. Used here as a grounding/faithfulness proxy:
      - High F1  -> answer's meaning is well supported by the retrieved context
      - Low F1   -> answer may be drifting from / hallucinating beyond the context

    Returns (precision, recall, f1) as plain floats.
    """
    if not reference_text.strip() or not candidate_text.strip():
        return 0.0, 0.0, 0.0

    # BERTScore compares sentence-by-sentence internally; long inputs are fine,
    # but we truncate extremely long context purely to keep this fast.
    ref = reference_text[:4000]
    cand = candidate_text[:4000]

    P, R, F1 = bert_score(
        [cand], [ref],
        lang="en",
        model_type="distilbert-base-uncased",
        verbose=False,
    )
    return P.item(), R.item(), F1.item()


def financial_rag_with_bertscore(query: str, company: str = None, session_id: str = "default"):
    """
    Thin wrapper around financial_rag() that additionally computes and prints
    a BERTScore for the generated response (answer vs. retrieved context),
    and logs it to _bert_log for later inspection / averaging.
    """
    response = financial_rag(query, company=company, session_id=session_id)

    context = getattr(financial_rag, "_last_context", "")
    precision, recall, f1 = compute_bertscore(context, response) if context else (0.0, 0.0, 0.0)

    print(f"📊 BERTScore (answer vs. retrieved context) — Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

    _bert_log.append({
        "query": query.strip()[:80],
        "company": company or "All",
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    })

    return response


# Backward-compatible alias, in case earlier cells still call the old name
financial_rag_with_bleu = financial_rag_with_bertscore


def show_bertscore_log():
    """Pretty-print the BERTScore for every query run so far."""
    if not _bert_log:
        print("No queries logged yet.")
        return
    print(f"{'#':<3} {'Company':<10} {'Precision':<10} {'Recall':<10} {'F1':<8} Query")
    print("-" * 100)
    for i, entry in enumerate(_bert_log, 1):
        print(f"{i:<3} {entry['company']:<10} {entry['precision']:<10} {entry['recall']:<10} {entry['f1']:<8} {entry['query']}")
    avg_p = sum(e['precision'] for e in _bert_log) / len(_bert_log)
    avg_r = sum(e['recall'] for e in _bert_log) / len(_bert_log)
    avg_f1 = sum(e['f1'] for e in _bert_log) / len(_bert_log)
    print("-" * 100)
    print(f"Average -> Precision: {avg_p:.4f} | Recall: {avg_r:.4f} | F1: {avg_f1:.4f}")

print("✅ BERTScore ready — use financial_rag_with_bertscore(query, company=...) to see scores")
print("✅ Call show_bertscore_log() anytime to see all scores so far")


✅ BERTScore ready — use financial_rag_with_bertscore(query, company=...) to see scores
✅ Call show_bertscore_log() anytime to see all scores so far


In [20]:
from IPython.display import display, Markdown

query = """Provide a comprehensive financial analysis of NVIDIA using the latest SEC 10-K filing.

Focus on:
- Total revenue breakdown and year-over-year growth (Data Center vs Gaming vs Professional Visualization vs Automotive)
- Data Center segment performance, including AI infrastructure demand drivers
- Discuss What are the Gross margin trends and key factors affecting profitability
- Discuss what are the Operating expenses, operating income, and net income trends
- Cash flow generation, capital expenditures, and liquidity position
- Key financial highlights and management commentary on future outlook


Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors."""



response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Provide a comprehensive financial analysis of NVIDIA using t...
   Retrieval Quality: 20/25 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 20/25 words matched


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

📊 BERTScore (answer vs. retrieved context) — Precision: 0.7605 | Recall: 0.7845 | F1: 0.7723


# Financial Analysis — Nvidia

**Financial Analysis: NVIDIA Corporation (Fiscal Year Ended January 25, 2026)**

### 1. Total Revenue Breakdown and Year-over-Year Growth

NVIDIA demonstrated robust top-line expansion in fiscal year 2026, with total revenue increasing significantly compared to fiscal year 2025. The growth was driven primarily by the Data Center segment, while Gaming and other segments contributed to a diversified revenue base.

| Metric | FY 2026 (Jan 25, 2026) | FY 2025 (Jan 26, 2025) | Change ($) | % Change |
| :--- | :--- | :--- | :--- | :--- |
| **Total Revenue** | **$215,938 million** | **$130,497 million** | **$85,441 million** | **65.5%** |
| Data Center | $193,737 million | $115,186 million | $78,551 million | 68.2% |
| Gaming | $16,042 million | $11,350 million | $4,692 million | 41.3% |
| Professional Visualization | $3,191 million | $1,878 million | $1,313 million | 70.0% |
| Automotive | $2,349 million | $1,694 million | $655 million | 38.7% |
| OEM and Other | $619 million | $389 million | $230 million | 59.1% |

*Calculations:*
*   Total Revenue % Change: $((215,938 - 130,497) / 130,497) \times 100 = 65.5\%$
*   Data Center % Change: $((193,737 - 115,186) / 115,186) \times 100 = 68.2\%$
*   Gaming % Change: $((16,042 - 11,350) / 11,350) \times 100 = 41.3\%$
*   Professional Visualization % Change: $((3,191 - 1,878) / 1,878) \times 100 = 70.0\%$
*   Automotive % Change: $((2,349 - 1,694) / 1,694) \times 100 = 38.7\%$

**Analysis:**
The Data Center segment is the primary engine of growth, accounting for approximately 89.7% of total revenue in FY 2026 ($193,738 million / $215,938 million). Within Data Center, Compute revenue was $162,361 million and Networking revenue was $31,376 million. The context notes that Data Center computing grew 59% driven by demand for the Blackwell computing platform. Gaming, while growing 41.3%, represents a smaller portion of the total mix (7.4%). Professional Visualization showed the highest percentage growth among major segments at 70.0%, though from a smaller base.

### 2. Data Center Segment Performance and AI Infrastructure Demand Drivers

The Data Center segment remains the core driver of NVIDIA’s financial performance.

*   **Revenue Contribution:** Data Center revenue reached $193,737 million in FY 2026, up from $115,186 million in FY 2025.
*   **Sub-segment Breakdown:**
    *   **Compute:** $162,361 million (FY 2026) vs. $102,196 million (FY 2025).
    *   **Networking:** $31,376 million (FY 2026) vs. $12,990 million (FY 2025).
*   **Demand Drivers:** The context explicitly states that revenue growth was driven by "data center compute and networking platforms for accelerated computing and AI solutions." The Blackwell architectures represented the majority of Data Center revenue.
*   **Strategic Positioning:** NVIDIA describes itself as a "data center scale AI infrastructure company reshaping all industries." The demand is fueled by the need for accelerated computing and AI-related cloud services.

### 3. Gross Margin Trends and Profitability Factors

NVIDIA maintained high gross margins, though there was a slight compression in FY 2026 compared to FY 2025, likely due to the mix shift toward high-volume Data Center products or specific cost structures.

| Metric | FY 2026 | FY 2025 | FY 2024 |
| :--- | :--- | :--- | :--- |
| **Gross Profit ($)** | **$153,463 million** | **$97,858 million** | **$44,301 million** |
| **Gross Margin (%)** | **71.1%** | **75.0%** | **72.7%** |

*Calculations:*
*   FY 2026 Gross Margin: $153,463 / 215,938 = 71.1\%$ (Matches context: 71.1%)
*   FY 2025 Gross Margin: $97,858 / 130,497 = 75.0\%$ (Matches context: 75.0%)
*   FY 2024 Gross Margin: $44,301 / 60,922 = 72.7\%$

**Analysis:**
While absolute Gross Profit increased by 56.8% ($((153,463 - 97,858) / 97,858) \times 100$), the Gross Margin percentage declined from 75.0% in FY 2025 to 71.1% in FY 2026. This 390 basis point decrease suggests that the cost of revenue (which increased from $32,639 million to $62,475 million) grew faster than revenue. The context notes that Cost of Revenue as a percentage of revenue increased from 25.0% to 28.9%. This margin compression is a key factor affecting profitability, even as absolute profits surged.

### 4. Operating Expenses, Operating Income, and Net Income Trends

NVIDIA exhibited strong operating leverage, with operating income growing faster than revenue, leading to high net income.

| Metric | FY 2026 | FY 2025 | FY 2024 | Change (FY26 vs FY25) |
| :--- | :--- | :--- | :--- | :--- |
| **Total Operating Expenses** | **$23,076 million** | **$16,405 million** | **$11,329 million** | **+40.7%** |
| **Operating Income** | **$130,387 million** | **$81,453 million** | **$32,972 million** | **+60.1%** |
| **Net Income** | **$120,067 million** | **$72,880 million** | **$29,760 million** | **+64.8%** |

*Calculations:*
*   Total Operating Expenses % Change: $((23,076 - 16,405) / 16,405) \times 100 = 40.7\%$
*   Operating Income % Change: $((130,387 - 81,453) / 81,453) \times 100 = 60.1\%$
*   Net Income % Change: $((120,067 - 72,880) / 72,880) \times 100 = 64.8\%$

**Expense Breakdown:**
*   **Research and Development (R&D):** Increased from $12,914 million to $18,497 million (43.2% increase). This reflects continued investment in technology roadmaps and AI ecosystem support.
*   **Sales, General and Administrative (SG&A):** Increased from $3,491 million to $4,579 million (31.2% increase).

**Analysis:**
Operating Income grew 60.1%, outpacing the 65.5% revenue growth slightly less, but significantly outpacing the 40.7% growth in operating expenses. This indicates strong operating leverage. The Operating Margin improved from 62.4% in FY 2025 to 60.4% in FY 2026? *Correction based on context:* The context table on page 57 lists Operating Income as 60.4% of revenue for FY 2026 and 62.4% for FY 2025. Let's verify:
*   FY 2026 Op Margin: $130,387 / 215,938 = 60.4\%$
*   FY 2025 Op Margin: $81,453 / 130,497 = 62.4\%$
The operating margin declined by 200 basis points, consistent with the gross margin compression. However, Net Income margin remained stable, declining only slightly from 55.8% to 55.6%. This stability is supported by "Other income, net," which surged from $1,034 million to $9,022 million, contributing significantly to the bottom line.

### 5. Cash Flow Generation, Capital Expenditures, and Liquidity Position

NVIDIA generated substantial cash from operations, which was partially offset by significant investing and financing activities.

| Metric | FY 2026 | FY 2025 |
| :--- | :--- | :--- |
| **Net Cash Provided by Operating Activities** | **$102,718 million** | **$64,089 million** |
| **Net Cash Used in Investing Activities** | **$(52,228) million** | **$(20,421) million** |
| **Net Cash Used in Financing Activities** | **$(48,474) million** | **$(42,359) million** |

**Liquidity Position:**
*   As of January 25, 2026, NVIDIA had **$62.6 billion** in cash,

## Sources
- Nvidia.pdf | Page 71
- Nvidia.pdf | Page 106
- Nvidia.pdf | Page 72
- Nvidia.pdf | Page 57
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 60
- Nvidia.pdf | Page 53
- Nvidia.pdf | Page 22

In [21]:
query = "Discuss about the architecture of Blackwell and Rubin and how they are benefits to Nvidia in depth "

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Discuss about the architecture of Blackwell and Rubin and ho...
   Retrieval Quality: 1/3 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 1/3 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.8150 | Recall: 0.8436 | F1: 0.8291


# Financial Analysis — Nvidia

### Financial Analysis: NVIDIA Blackwell and Rubin Architectures

**Question Type:** B (Architecture, Technology, Strategy)
**Action:** Direct structured analysis based on Context. No financial table required.

#### 1. NVIDIA Blackwell Architecture
**Launch and Composition**
NVIDIA launched the Blackwell architecture in fiscal year 2025. It is defined as a full set of data center scale infrastructure that includes GPUs, CPUs, DPUs, interconnects, switch chips and systems, and networking adapters. In 2024, the architecture was specifically noted for connecting 36 Grace CPUs and 72 Blackwell GPUs in a data center scale, liquid-cooled design.

**Performance and Application**
Blackwell is designed to excel at processing cutting-edge generative AI and accelerated computing workloads. It delivers market-leading performance and efficiency. The architecture is offered in a number of configurations to serve customers across industries for diverse AI and accelerated computing use cases. Specifically, the 2024 iteration was built for real-time trillion-parameter inference and training.

**Strategic Benefit to NVIDIA**
*   **Revenue Driver:** In fiscal year 2026, revenue growth was driven by data center compute and networking platforms for accelerated computing and AI solutions, with Blackwell architectures representing the majority of Data Center revenue.
*   **Platform Expansion:** The architecture supports NVIDIA’s transition into a data center scale AI infrastructure company, leveraging a unified underlying architecture across GPUs, CPUs, CUDA, and networking technologies.
*   **Full-Stack Ecosystem:** Blackwell integrates with NVIDIA’s software stacks, including NVIDIA AI Enterprise, which includes NVIDIA NIM, NeMo, and AI Blueprints, allowing for the deployment of production-grade generative AI applications.

#### 2. NVIDIA Rubin Platform
**Launch and Timeline**
NVIDIA unveiled the Rubin platform in fiscal year 2026. Production shipments are expected to commence in the second half of fiscal year 2027.

**Performance and Application**
Rubin is built specifically for agentic AI and reasoning. It excels at processing multi-step problem-solving and massive long-context workflows. A key performance metric disclosed is that Rubin delivers up to a 10x reduction in cost per token compared to Blackwell.

**Strategic Benefit to NVIDIA**
*   **Cost Efficiency Leadership:** The 10x reduction in cost per token relative to Blackwell reinforces NVIDIA’s position in delivering order-of-magnitude performance advantages relative to legacy approaches.
*   **Product Cadence:** Rubin is part of NVIDIA’s strategy to bring new advanced architectures on a one-year product cadence, ensuring continuous technological leadership.
*   **Agentic AI Focus:** By targeting agentic AI and reasoning, Rubin aligns with the evolving demand for complex, multi-step AI workflows, securing NVIDIA’s role in the next phase of AI adoption.

#### 3. Comparative Strategic Impact
*   **Performance Trajectory:** The progression from Blackwell to Rubin demonstrates NVIDIA’s ability to outpace Moore’s Law through innovation across architecture, chip design, system, interconnect, algorithm, and software layers.
*   **Market Positioning:** Both architectures support NVIDIA’s two operating segments, "Compute & Networking" and "Graphics," but primarily drive the "Compute & Networking" segment through data center scale solutions.
*   **Risk Mitigation:** While the complexity of product transitions may cause delays in production and supply/demand management challenges, the sustained demand for exceptional 3D graphics and AI infrastructure supports the continued scale of these platforms.

**Note on Financial Metrics:** Specific revenue figures, operating expenses, or net income attributable solely to Blackwell or Rubin are not disclosed in the retrieved sections. The only quantitative performance metric provided is the "up to a 10x reduction in cost per token" for Rubin compared to Blackwell.

## Sources
- Nvidia.pdf | Page 7
- Nvidia.pdf | Page 4
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 9

In [22]:

# CELL 26: NVIDIA — Export Control Risks

from IPython.display import display, Markdown

query = """Provide a detailed analysis of NVIDIA's export control risks, geopolitical exposure, and China-related challenges.

Focus on:
- Impact of U.S. export restrictions on products
- Licensing requirements, revenue impact, and inventory charges
- Competitive effects on China data center market discuss this in detail
- Mitigation strategies and long-term implications
- Broader supply chain and regulatory risks discuss this in detail

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 


Quote relevant sections from the Risk Factors and Business sections."""

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Provide a detailed analysis of NVIDIA's export control risks...
   Retrieval Quality: 20/23 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 20/23 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7954 | Recall: 0.8115 | F1: 0.8034


# Financial Analysis — Nvidia

**Financial Analysis: NVIDIA Export Control Risks, Geopolitical Exposure, and China-Related Challenges**

Based on the provided Risk Factors and Business sections from NVIDIA’s 10-K filing (Fiscal Year 2026), the following is a detailed analysis of the company’s exposure to export controls, geopolitical tensions, and regulatory headwinds.

### 1. Impact of U.S. Export Restrictions on Products

NVIDIA faces significant operational and strategic constraints due to unilateral worldwide controls imposed by the United States Government (USG) on GPUs and associated products. These restrictions are not limited to a single region but have a global scope, creating a complex compliance landscape.

*   **Scope of Restrictions:** The USG has imposed controls that restrict the export of NVIDIA’s technology, products, and services. These controls are described as "very broad in scope and application," potentially prohibiting exports to any or all customers in one or more markets.
*   **Specific Product Targets:** The restrictions specifically target data center products. In January 2025, the USG published the AI Diffusion Interim Final Rule (IFR) in the Federal Register, which would have imposed a worldwide licensing requirement on NVIDIA’s data center products, including the **H200, GB200, and GB300**. Although the USG announced in May 2025 that it would rescind the AI Diffusion IFR and implement a replacement rule, the scope, timing, and requirements of the forthcoming rule remain uncertain.
*   **Operational Disruption:** Export controls disrupt NVIDIA’s supply and distribution chains. A substantial portion of NVIDIA’s products are warehoused in and distributed from **Hong Kong**. Restrictions on selling data center GPUs also negatively impact demand for networking products used in servers containing those GPUs. Furthermore, the USG may impose export controls on networking products, such as high-speed network interconnects, to limit the ability of downstream parties to create large clusters for frontier model training.

> **Quote:** "The United States has imposed unilateral worldwide controls restricting GPUs and associated products, and it is likely that additional unilateral or multilateral controls will be adopted. Such controls have been and may again be very broad in scope and application, prohibit us from exporting our products to any or all customers in one or more markets, and could negatively impact our manufacturing, testing and warehousing locations and options..."

### 2. Licensing Requirements, Revenue Impact, and Inventory Charges

The licensing process associated with export controls introduces significant uncertainty and financial risk, directly impacting revenue recognition and inventory valuation.

*   **Licensing Uncertainty:** Even if the USG grants requested licenses, they may be temporary and impose burdensome conditions regarding installation, maintenance, and use. These licenses may include financial or economic requirements that NVIDIA, its customers, or end users cannot or choose not to fulfill. The licensing process may not be resolved before significant business opportunities evaporate.
*   **Revenue Impact:** Export controls have already negatively impacted demand for NVIDIA’s products and services, not only in China but also in other markets such as **Europe, Latin America, and Southeast Asia**. The possibility of additional export controls has negatively impacted demand, benefiting competitors that offer alternatives less likely to be restricted.
*   **Inventory and Supply Charges:** Reduced demand due to export controls has led to and could in the future lead to **excess inventory** or cause NVIDIA to incur related supply charges. The text specifically references a recent experience with the **H20** product, where new unilateral export controls restricted its sale by the time it was ready for market, resulting in excess inventory and purchase obligations.
*   **Tariff Pass-Through Risk:** If NVIDIA is able to sell licensed products into the China market, it may not be able to pass along all or any of the tariff to its customers, potentially leading to increased costs and a harmed competitive position.

> **Quote:** "Export controls increase the risk of investing in U.S. advanced semiconductor products, because by the time a new product is ready for market, it may be subject to new unilateral export controls restricting its sale, resulting in excess inventory and purchase obligations as we recently experienced with the H20."

### 3. Competitive Effects on China Data Center Market

NVIDIA’s position in the China data center market has been severely compromised, leading to a structural shift in competitive dynamics that favors rivals.

*   **Effective Foreclosure:** As of the end of fiscal year 2026, NVIDIA was **effectively foreclosed from competing in China's data center computing/compute market**. This is due to the inability to create and deliver a competitive product that receives approval from both the USG and the Chinese government.
*   **Benefit to Competitors:** The effective foreclosure from the China market has helped competitors build larger developer and customer ecosystems to challenge NVIDIA worldwide. The Chinese government has encouraged customers to purchase from China-based competitors and discouraged customers from purchasing, importing, or using NVIDIA’s data center products, including any China-specific product designed to comply with U.S. export controls.
*   **Design-Out Risk:** Export controls have encouraged and may in the future encourage customers outside China and other impacted regions to “design-out” certain U.S. semiconductors from their products to reduce compliance burden and risk. This ensures customers can serve markets worldwide without restriction.
*   **Mellanox Acquisition Violation:** On **September 15, 2025**, China’s antitrust regulators published a preliminary finding that NVIDIA’s compliance with applicable U.S. export controls, which required offering degraded products to the Chinese market, discriminated unfairly against customers in the China market. This was found to violate the terms of China’s approval of NVIDIA’s **Mellanox acquisition**. If regulators conclude that NVIDIA has failed to fulfill these terms or violated applicable law in China, the company could be subject to financial penalties, restrictions on its ability to conduct business, or other orders regarding its networking business, products, and services.

> **Quote:** "As of the end of fiscal year 2026, we were effectively foreclosed from competing in China's data center computing/compute market, and our effective foreclosure from the China market helped our competitors build larger developer and customer ecosystems to challenge us worldwide."

### 4. Mitigation Strategies and Long-Term Implications

NVIDIA is actively working to mitigate these risks, though the long-term implications remain material and adverse if unresolved.

*   **Supply Chain Resiliency:** NVIDIA is working to enhance the resiliency and redundancy of its supply chain, which is currently concentrated in **Asia**. However, new and existing export controls could limit alternative manufacturing locations, negatively impacting the business.
*   **Product Adaptation:** NVIDIA has attempted to offer degraded products to the Chinese market to comply with U.S. export controls. However, this strategy has led to the antitrust violation finding by Chinese regulators, suggesting that compliance with U.S. rules may inherently conflict with Chinese regulatory requirements.
*   **Long-Term Implications:** Unless NVIDIA is able to return to the China market with a product that meets the approval of both the USG and the Chinese government, the lost opportunity and the benefit to competitors will have a **material and adverse impact** on NVIDIA’s business, operating results, and financial condition. The text emphasizes that export controls have a disproportionate impact on NVIDIA compared to competitors selling chips outside the scope of such controls.

> **Quote:** "While we work to enhance the resiliency and redundancy of our supply chain, which is currently concentrated in Asia, new and existing export controls or changes to existing export controls could limit alternative manufacturing locations and negatively impact our business."

### 5. Broader Supply Chain and Regulatory Risks

Beyond China, NVIDIA faces a broad spectrum of regulatory and supply chain risks globally.

*   **Global Regulatory Scrutiny:** NVIDIA’s position in AI markets has led to increased interest from regulators worldwide, including the **European Union, the United States, the United Kingdom, South Korea, Japan, and China**. For example, the French Competition Authority has collected information regarding NVIDIA’s business and competition in the graphics card and CSP market. NVIDIA has received broad requests for information from competition regulators in these regions regarding GPU sales, supply allocation, foundation models, and partnerships.
*   **AI-Specific Restrictions:** Concerns regarding the misuse of AI applications have resulted in unilateral or multilateral restrictions on products used for training, modifying, tuning, and deploying Large Language Models (LLMs). These restrictions limit the ability of downstream customers worldwide to acquire, deploy, and use systems including NVIDIA’s products.
*   **Geopolitical Conflicts:** The war in **Ukraine** has impacted sales in the **EMEA** (Europe, Middle East, and Africa) region and may continue to do so.
*   **Compliance Burdens:** Repeated changes in export control rules impose compliance burdens on NVIDIA and its customers, negatively and materially impacting the business. Additional controls may include deemed export control limitations that negatively impact the ability of NVIDIA’s research and development teams to execute its roadmap in a timely manner.
*   **Foreign Government Retaliation:** Additional export restrictions may provoke responses from foreign governments, including China, that negatively impact NVIDIA’s supply chain or its ability to provide products and services to customers in all markets worldwide, which could substantially reduce revenue.

> **Quote:** "Our position in markets relating to AI has led to increased interest in our business from regulators worldwide, including the European Union, the United States, the United Kingdom, South Korea, Japan, and China."

### Investor Advice

Based on the detailed risk factors disclosed, investors should consider the following actions and considerations:

1.  **Assess Revenue Concentration Risk:** Investors should closely monitor NVIDIA’s revenue breakdown by geography. Given the "effective foreclosure" from the China data center market as of fiscal year 2026, any historical revenue reliance on China is likely to be permanently impaired. Investors should adjust valuation models to reflect a lower long-term revenue ceiling in the Asia-Pacific region, excluding China.
2.  **Monitor Regulatory Developments:** The uncertainty surrounding the replacement rule for the AI Diffusion IFR (announced for rescission in May 2025) is a critical variable. Investors should track Federal Register publications and USG announcements for the scope of the new rule. A broader scope could further restrict sales in Tier 2 countries, impacting global data center growth.
3.  **Evaluate Competitive Ecosystem Shifts:** The text explicitly states that NVIDIA’s foreclosure from China has helped competitors build larger developer and customer ecosystems. Investors should analyze the market share gains of competitors (such as AMD, Intel, or Chinese domestic firms) in the AI GPU space. If these ecosystems become entrenched, NVIDIA may face a permanent loss of market share even if export controls are lifted.
4.  **Watch for Financial Penalties and Litigation:** The preliminary finding by China’s antitrust regulators regarding the Mellanox acquisition terms poses a direct risk of financial penalties and operational restrictions. Investors should review NVIDIA’s legal disclosures for updates on this case and assess the potential magnitude of fines or business restrictions.
5.  **Supply Chain Diversification Progress:** Given the concentration of the supply chain in Asia and the risks associated with export controls, investors should evaluate NVIDIA’s progress in diversifying manufacturing and warehousing locations. Delays in this diversification could lead to further inventory charges and supply disruptions.
6.  **Compliance Cost Impact:** The increasing compliance burdens from repeated changes in export control rules will likely increase operating expenses. Investors should factor in higher SG&A or R&D costs related to compliance and legal defense in their earnings forecasts.

In summary, while NVIDIA remains a dominant player in AI semiconductors, the geopolitical and regulatory environment presents material, long-term risks to its revenue growth, market share, and profitability. Investors should adopt a conservative stance on growth assumptions for the China market and closely monitor regulatory developments in the U.S. and allied nations.

## Sources
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 39
- Nvidia.pdf | Page 15
- Nvidia.pdf | Page 41
- Nvidia.pdf | Page 37

In [23]:
query = """Evaluate NVIDIA's  positioning across its markets.

Focus on:
- Competition in Data Center (AMD, Intel, custom ASICs from hyperscalers)
- Discuss about Gaming GPU competition 
- Professional Visualization and Automotive segments
- Overall technology leadership in GPUs, CUDA, networking, and software
- Barriers to entry and ecosystem strength

Discuss strengths, weaknesses, and investor implications with references from the filing."""

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Evaluate NVIDIA's  positioning across its markets.

Focus on...
   Retrieval Quality: 18/14 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 18/14 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.8003 | Recall: 0.8302 | F1: 0.8150


# Financial Analysis — Nvidia

**Financial Analysis: NVIDIA Market Positioning and Competitive Landscape**

Based on the retrieved sections of the NVIDIA Annual Report, the following is an evaluation of the company’s positioning across its key markets, technology leadership, and competitive risks.

### 1. Data Center and AI Infrastructure
NVIDIA positions itself as a "data center scale AI infrastructure company" that provides a complete, end-to-end accelerated computing platform for AI, addressing both training and inferencing.

*   **Technology Leadership:** The company leverages a full-stack innovation approach across architecture, chip design, system, interconnect, algorithm, and software layers. This allows NVIDIA to deliver "order-of-magnitude performance advantages relative to legacy approaches."
*   **Product Portfolio:** NVIDIA offers all three major processing units in AI servers: GPUs, CPUs, and DPUs. The filing notes that "GPUs are uniquely suited to AI," and the company continues to add AI-specific features to its GPU architecture.
*   **Recent Performance Driver:** Revenue growth in fiscal year 2026 was driven by data center compute and networking platforms. Specifically, "Blackwell architectures represented the majority of our Data Center revenue."
*   **Software Ecosystem:** NVIDIA AI Enterprise is a comprehensive software suite for production-grade generative AI, including NVIDIA NIM (for token throughput), NVIDIA NeMo (for model curation and fine-tuning), and AI Blueprints (pre-built templates for AI agents).
*   **Market Dominance:** Including GPUs and networking, NVIDIA powers "over 78% of the supercomputers on the global TOP500 list," including "9 of the top 10 systems on the Green500 list."

### 2. Competition in Data Center
The filing explicitly identifies several sources of competition for Data Center and accelerated computing products:
*   **Direct Competitors:** Advanced Micro Devices, Inc. (AMD), Huawei Technologies Co. Ltd. (Huawei), and Intel Corporation (Intel) are cited as suppliers of hardware and software for discrete and integrated GPUs, custom chips, and other accelerated computing solutions.
*   **Hyperscaler Custom ASICs:** Large cloud services companies with internal teams designing hardware and software that incorporate accelerated or AI computing functionality are listed as competitors. These include Alibaba Group, Alphabet Inc., Amazon, Inc. (Amazon), Baidu, Inc., Huawei, and Microsoft Corporation (Microsoft).
*   **Networking Competition:** Competitors for networking products (switches, network adapters/DPUs, cable solutions) include AMD, Arista Networks, Broadcom, Cisco Systems, Inc., Hewlett Packard Enterprise Company, Huawei, Intel, Lumentum Holdings Inc., and Marvell Technology, Inc.
*   **Competitive Pressure:** The company expects competition to increase from both existing competitors and new market entrants. Some competitors may have "greater marketing, financial, distribution and manufacturing resources" and may offer products that are "lower priced than ours or may provide better performance or additional features."

### 3. Gaming GPU Competition
*   **Market Position:** NVIDIA serves the gaming market with GeForce RTX GPUs for desktop and laptop PCs, the GeForce NOW cloud gaming service, and SoCs for game consoles.
*   **Technology:** The platform leverages GPUs and sophisticated software, including ray tracing and Deep Learning Super Sampling (DLSS). In fiscal year 2025, NVIDIA announced the "NVIDIA Blackwell GeForce RTX 50 Series family," which introduced "neural graphics" combining AI models with traditional rendering.
*   **Competitive Landscape:** While specific gaming competitors are not listed in the same granular detail as Data Center competitors, the general competitive risk applies. The filing notes that competition comes from suppliers of hardware and software for discrete and integrated GPUs, including AMD, Huawei, and Intel.
*   **Growth Drivers:** Growth is propelled by high production value games, eSports, social connectivity, and the popularity of streamers and creators. The market is also expanding due to the growing population of live streamers, broadcasters, artists, and creators.

### 4. Professional Visualization and Automotive
*   **Professional Visualization:** NVIDIA works closely with Independent Software Vendors (ISVs) to optimize offerings for NVIDIA GPUs. The RTX PRO GPUs are designed for design, engineering, and digital content creation. The filing notes that "generative and agentic AI is expanding the market for our workstation-class GPUs," as enterprise customers deploy AI applications on-premises. These GPUs leverage the same Tensor Core technology found in Data Center solutions.
*   **Automotive:** The filing mentions that NVIDIA addresses the Automotive market with a unified underlying architecture leveraging GPUs, CPUs, CUDA, and networking technologies. It also references SoC products used in "servers or embedded into automobiles, autonomous machines, and gaming devices." Competitors in the SoC space include Ambarella, Inc., AMD, Broadcom, Intel, Qualcomm Incorporated, Renesas Electronics Corporation, Samsung, and Tesla, Inc.

### 5. Overall Technology Leadership and Ecosystem Strength
*   **Unified Architecture:** NVIDIA addresses diverse end markets (Data Center, Gaming, Professional Visualization, Automotive) with a "unified underlying architecture" leveraging GPUs, CPUs, CUDA, and networking technologies as fundamental building blocks.
*   **Platform Approach:** The programmable nature of the architecture allows for "leveraged investments in research and development," enabling support for several multi-billion-dollar end markets with shared underlying technology.
*   **Ecosystem:** The company’s AI technology leadership is "reinforced by our large and expanding ecosystem." This includes support for "6,000 applications" and partnerships with ISVs and third-party developers.
*   **Performance Advantage:** The full-stack innovation approach allows NVIDIA to deliver "order-of-magnitude performance advantages" and "continued performance leaps that outpace Moore’s Law."

### 6. Barriers to Entry and Risks
*   **Intellectual Property:** NVIDIA relies on patents, trademarks, trade secrets, nondisclosure agreements, and licensing arrangements to protect its IP.
*   **Regulatory and Export Controls:**
    *   **Regulatory Scrutiny:** NVIDIA’s position in AI has led to increased interest from regulators worldwide, including the European Union, the United States, the United Kingdom, South Korea, Japan, and China. The company has received broad requests for information regarding its sales, supply allocation, and partnerships.
    *   **Export Controls:** Export controls may disrupt the supply and distribution chain, particularly for products warehoused in Hong Kong. Controls restricting the sale of data center GPUs may negatively impact demand for networking products. The filing notes that export controls have "already and may in the future encourage customers outside China and other impacted regions to 'design-out' certain U.S. semiconductors."
    *   **Competitive Disadvantage:** Export controls may disadvantage NVIDIA against competitors selling chips outside the scope of such controls.
*   **Resource Constraints:** The availability of data centers, energy, and capital to support the buildout of NVIDIA AI infrastructure is crucial. Shortages in these resources could impact future revenue. Expanding energy capacity is described as a "complex, multi-year process" with significant regulatory, technical, and construction challenges.

### Investor Implications
*   **Strengths:**
    *   Dominant market share in supercomputing (78% of TOP500).
    *   Full-stack technology leadership (hardware, software, networking).
    *   Strong ecosystem with 6,000 supported applications.
    *   Revenue growth driven by high-demand Data Center AI solutions (Blackwell architecture).
*   **Weaknesses/Risks:**
    *   Intense competition from AMD, Intel, Huawei, and hyperscalers (Amazon, Microsoft, etc.) who may have greater resources or offer lower-priced alternatives.
    *   Significant regulatory and export control risks, particularly regarding China and other regions, which could lead to "design-out" of NVIDIA products.
    *   Dependence on external factors such as energy capacity and capital availability for customer infrastructure buildout.
    *   Potential for new competitors or alliances to acquire significant market share.

In summary, NVIDIA maintains a strong position in Data Center AI and Gaming through its full-stack technology and ecosystem. However, investors should note the significant competitive pressures from both traditional semiconductor peers and hyperscaler custom silicon, as well as the material risks posed by regulatory scrutiny and export controls.

## Sources
- Nvidia.pdf | Page 9
- Nvidia.pdf | Page 12
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 7
- Nvidia.pdf | Page 5

In [24]:
query = """Analyze NVIDIA's long-term corporate strategy, key risks, and growth outlook.

Focus on:
- Platform strategy (hardware + software + ecosystem)
- Expansion into AI, robotics, autonomous driving, and professional visualization
- Supply chain, manufacturing, and capacity risks
- Human capital, R&D investment, and innovation approach
- Major risks from the Risk Factors section and mitigation efforts

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 


Provide a balanced view with exact quotes and key takeaways for long-term investors."""


response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Analyze NVIDIA's long-term corporate strategy, key risks, an...
   Retrieval Quality: 22/22 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 22/22 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7804 | Recall: 0.7832 | F1: 0.7818


# Financial Analysis — Nvidia

**Financial Analysis: NVIDIA Long-Term Corporate Strategy, Risks, and Growth Outlook**

**Classification:** Type B (Strategy, Technology, and Risk Analysis)
*Note: As this question focuses on qualitative strategy, technology architecture, and risk factors rather than specific financial line items (Revenue, Net Income, etc.), no financial table is generated. All analysis is derived strictly from the provided Context.*

### 1. Platform Strategy: Hardware, Software, and Ecosystem Integration

NVIDIA’s core strategic advantage lies in its transition from a pure hardware provider to a "data center scale AI infrastructure company" that reshapes industries. The company employs a **full-stack innovation approach** that leverages innovation across architecture, chip design, system, interconnect, algorithm, and software layers.

*   **Unified Architecture:** NVIDIA addresses diverse end markets (Data Center, Gaming, Professional Visualization, and Automotive) with a "unified underlying architecture leveraging our GPUs, CPUs, CUDA and networking technologies as the fundamental building blocks." This allows the company to support "several multi-billion-dollar end markets with shared underlying technology."
*   **Software Ecosystem:** The strategy extends beyond silicon to include "NVIDIA AI Enterprise," a comprehensive software suite for generative AI. Key components include:
    *   **NVIDIA NIM:** Increases token throughput using open and proprietary models.
    *   **NVIDIA NeMo:** A solution for curating, fine-tuning, and evaluating domain-adapted models.
    *   **AI Blueprints:** Pre-built templates for building and deploying AI agents.
*   **Ecosystem Expansion:** The company aims to "expand the ecosystem for our products and technologies," relying on a "large and expanding ecosystem" to reinforce its technology leadership. This includes partnerships with third-party developers and partners who utilize NVIDIA’s software stacks.

**Key Takeaway:** NVIDIA is not selling chips in isolation; it is selling an integrated platform where hardware performance is amplified by proprietary software and a broad developer ecosystem, creating high switching costs for customers.

### 2. Expansion into AI, Robotics, Autonomous Driving, and Visualization

NVIDIA has expanded its focus from its "original focus on PC graphics" into "several other large and important computationally intensive fields."

*   **AI and Data Center:** The company states it is "now a data center scale AI infrastructure company." Revenue growth in fiscal year 2026 was "driven by data center compute and networking platforms for accelerated computing and AI solutions." The **Blackwell architectures** are highlighted as representing "the majority of our Data Center revenue." NVIDIA provides "full-stack data center-scale compute and networking solutions across processing units, interconnects, systems, and software," including GPUs, CPUs, and DPUs.
*   **Autonomous Driving and Robotics:** The company leverages its GPU architecture for "autonomous vehicles, robotics, and digital twin applications." Specific products include the **DRIVE** platform, which features:
    *   An in-vehicle operating system (DRIVE OS).
    *   A reference sensor set supporting "full self-driving capability."
    *   An open, modular software platform for autonomous driving, mapping, and parking.
*   **Professional Visualization and Scientific Computing:** NVIDIA has leveraged its GPU architecture to create platforms for "scientific computing, AI, data science," and professional visualization, addressing problems that were "previously deemed unsolvable."

**Key Takeaway:** The diversification from gaming into AI, data centers, and autonomous systems reduces reliance on a single consumer market and positions NVIDIA at the center of multiple high-growth industrial sectors.

### 3. Supply Chain, Manufacturing, and Capacity Risks

The Context explicitly identifies significant risks related to the physical infrastructure required to support NVIDIA’s growth.

*   **Energy and Data Center Availability:** The text states that "The availability of data centers, energy, and capital to support the buildout of NVIDIA AI infrastructure by our customers and partners is crucial." Any shortage of these resources "could impact our future revenue and financial performance."
*   **Energy Constraints:** "Expanding energy capacity to meet demand is a complex, multi-year process that involves significant regulatory, technical, and construction challenges."
*   **Capital Access:** Access to capital is noted as "particularly constrained for less-capitalized companies," which may face difficulties securing financing for large-scale infrastructure projects. This suggests that while NVIDIA has the capital, its customers (especially smaller ones) may face bottlenecks in deploying NVIDIA’s hardware due to external energy and financing constraints.

**Key Takeaway:** The primary bottleneck for NVIDIA’s growth is no longer just chip production, but the external infrastructure (energy and data centers) required by its customers to deploy the hardware.

### 4. Human Capital, R&D Investment, and Innovation Approach

NVIDIA’s innovation model is characterized by high R&D intensity and a focus on outpacing traditional semiconductor scaling laws.

*   **R&D Strategy:** The company invests in R&D to "develop or secure access to new products and technologies." It notes that it has "invested in research and development in markets where we have a limited operating history, which may not produce meaningful revenue for several years, if at all." This indicates a long-term investment horizon for new verticals.
*   **Performance Advantage:** NVIDIA aims to deliver "continued performance leaps that outpace Moore’s Law." It claims to deliver "order-of-magnitude performance advantages relative to legacy approaches" in its target markets.
*   **Leveraged Investments:** The "programmable nature of our architecture allows us to make leveraged investments in research and development," enabling the company to support multiple end markets with shared technology.

**Key Takeaway:** NVIDIA’s R&D is structured to maximize ROI across multiple markets through a shared platform, but it carries the risk of long gestation periods for new product lines before revenue materialization.

### 5. Major Risks from Risk Factors and Mitigation

The Risk Factors section highlights several critical threats to NVIDIA’s financial results.

*   **Technological Obsolescence:** There is a risk of "failure to meet the evolving needs of our industry and markets." The company must "timely identify industry changes, adapt our strategies, and develop new or enhance and maintain existing products." Failure to do so could result in products becoming "incompatible with those developed by other companies" due to "disruptive technological innovations."
*   **Regulatory and Antitrust Risks:** NVIDIA’s position in AI has led to "increased interest in our business from regulators worldwide," including the EU, US, UK, South Korea, Japan, and China.
    *   **Specific Actions:** The French Competition Authority collected information regarding the graphics card and CSP market.
    *   **Broad Inquiries:** NVIDIA has received "broad requests for information" from regulators in the EU, US, UK, China, and South Korea regarding GPU sales, supply allocation, foundation model investments, and partnerships.
    *   **Compliance Costs:** The company faces "increased compliance costs as a result of changes or increases in antitrust legislation, regulation, administrative rule making, increased focus from regulators on cybersecurity vulnerabilities and risks."
*   **Customer and Industry Standards:** NVIDIA must meet "evolving and prevailing customer and industry safety, security, reliability expectations, and compliance standards."

**Mitigation Efforts:**
*   **Full-Stack Innovation:** By controlling the stack from hardware to software, NVIDIA aims to maintain "order-of-magnitude performance advantages."
*   **Ecosystem Expansion:** Expanding the ecosystem helps lock in customers and developers.
*   **Diversification:** Expanding into multiple verticals (AI, Automotive, Visualization) reduces dependence on any single market’s regulatory or technological shifts.

### 6. Advice for Long-Term Investors

Based on the provided Context, the following advice is offered to long-term investors:

1.  **Monitor Infrastructure Bottlenecks, Not Just Chip Supply:** Investors should closely track the availability of **energy capacity** and **data center construction** in key markets. Since "expanding energy capacity... is a complex, multi-year process," any delays in customer-side infrastructure could cap NVIDIA’s revenue growth despite strong product demand.
2.  **Assess Regulatory Exposure:** Given the "broad requests for information" from regulators in major economies (US, EU, China, etc.), investors should monitor antitrust developments. Increased compliance costs and potential restrictions on "supply allocation" or "partnerships" could impact margins and market access.
3.  **Evaluate R&D ROI in New Verticals:** The Context notes that R&D in markets with "limited operating history" may not produce meaningful revenue for "several years, if at all." Investors should be patient with the Automotive and Robotics segments, understanding that these are long-term plays rather than immediate profit drivers.
4.  **Focus on Ecosystem Stickiness:** The strength of NVIDIA’s "large and expanding ecosystem" and its software suite (NVIDIA AI Enterprise) is a key moat. Investors should look for signs of continued adoption of NVIDIA’s software tools (NIM, NeMo, Blueprints) as an indicator of long-term customer retention.
5.  **Watch for Technological Disruption:** The risk of "disruptive technological innovations" that could render NVIDIA’s products "incompatible" is a permanent threat. Investors should monitor NVIDIA’s ability to "outpace Moore’s Law" and maintain its "order-of-magnitude performance advantages" relative to competitors.

**Balanced View:**
NVIDIA presents a compelling long-term growth story driven by its dominant position in AI infrastructure and its full-stack platform strategy. However, this growth is not without significant headwinds. The company faces substantial regulatory scrutiny globally, potential bottlenecks in energy and data center capacity that are outside its direct control, and the perpetual risk of technological obsolescence in a rapidly evolving industry. Long-term investors must weigh the massive upside from AI adoption against the tangible risks of regulatory intervention and infrastructure constraints.

## Sources
- Nvidia.pdf | Page 18
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 9

In [25]:
query = "explain the supply chain risk of Nvidia in more depth like who are the key suppliers and what they supply and why it is cruical for Nvidia and also tell based on context suggest improvements in depth."

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: explain the supply chain risk of Nvidia in more depth like w...
   Retrieval Quality: 10/7 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 10/7 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7475 | Recall: 0.7845 | F1: 0.7655


# Financial Analysis — Nvidia

**Financial Analysis:**

**Classification:** Type B (Non-numeric topic: Supply Chain Risk, Strategy, and Operations).
*Note: The provided Context does not contain specific financial figures (Revenue, Expenses, etc.) for the supply chain section, nor does it list specific supplier names (e.g., TSMC, Samsung, SK Hynix). Therefore, no financial table is generated. The analysis below is derived strictly from the qualitative risk disclosures in the provided text.*

### 1. Supply Chain Risk Assessment

Based on the retrieved sections, Nvidia’s supply chain risk is characterized by **geopolitical vulnerability**, **inventory obsolescence**, and **regulatory compliance burdens**. The context highlights that the supply chain is not merely a logistical issue but a primary driver of financial volatility and competitive disadvantage.

#### A. Key Supply Chain Vulnerabilities Identified in Context

While the specific names of suppliers are **not disclosed in the retrieved sections**, the context identifies the *nature* of the supply chain dependencies and the specific products involved:

1.  **Geographic Concentration (Hong Kong):**
    *   **Risk:** A "substantial portion" of Nvidia’s products are **warehoused in and distributed from Hong Kong**.
    *   **Impact:** Export controls may disrupt this specific supply and distribution chain, creating a single point of failure for global logistics.

2.  **Product-Specific Supply Constraints:**
    *   **Data Center GPUs & Networking Products:** The context notes that restrictions on selling data center GPUs negatively impact demand for **networking products** (such as high-speed network interconnects) used in servers containing those GPUs.
    *   **Gaming GPUs:** As performance increases, export controls may have a "greater impact" on the ability to compete in controlled markets.
    *   **H20 Chip:** The context explicitly cites the **H20** as a recent example where new unilateral export controls resulted in **excess inventory and purchase obligations**.

3.  **Supplier & Counterparty Risks:**
    *   **Insolvency Risk:** The context lists the "insolvency of key suppliers, distributors, customers, CSPs, data center providers, licensing parties or other third parties" as a material risk.
    *   **Delivery Commitments:** There is a risk of "inability of our suppliers to deliver on their supply commitments," which directly impacts Nvidia’s ability to meet customer demand.
    *   **Financial Institution Instability:** Failures of counterparties (banks/insurers) could lead to market-wide liquidity problems, impacting vendors' ability to fulfill contractual obligations to Nvidia.

#### B. Why This is Crucial for Nvidia

The context establishes that supply chain integrity is critical for the following reasons:

1.  **Revenue Protection:** Export controls have "in the past and may in the future negatively impact demand for our products and services not only in China, but also in other markets, such as Europe, Latin America, and Southeast Asia."
2.  **Competitive Disadvantage:** Export controls create a "disproportionate impact on NVIDIA" compared to competitors whose chips are "outside the scope of such control." This allows foreign competitors to gain market share in regions where Nvidia is restricted.
3.  **Cost & Margin Erosion:**
    *   **Excess Inventory:** New controls can render ready-to-market products unsellable (e.g., H20), leading to write-downs and excess inventory.
    *   **Compliance Burdens:** Repeated changes in export control rules impose "compliance burdens" that negatively and materially impact the business.
    *   **Supply Charges:** Reduced demand due to controls can cause Nvidia to "incur related supply charges."
4.  **R&D Roadmap Disruption:** Additional unilateral or multilateral controls include "deemed export control limitations" that negatively impact the ability of R&D teams to execute the roadmap in a timely manner.

### 2. Suggested Improvements & Strategic Mitigations

Based *strictly* on the risks outlined in the context, the following improvements are suggested to mitigate these specific vulnerabilities:

#### A. Diversify Warehousing and Distribution Hubs
*   **Current Risk:** Heavy reliance on Hong Kong for warehousing and distribution.
*   **Suggested Improvement:** Decentralize the warehousing network to reduce exposure to single-region geopolitical shocks. Establishing alternative distribution hubs in regions less susceptible to the specific export control regimes mentioned (e.g., diversifying away from sole reliance on Hong Kong for global distribution) would mitigate the risk of "disrupting our supply and distribution chain."

#### B. Dynamic Inventory Management for Export-Controlled Products
*   **Current Risk:** Excess inventory and purchase obligations due to sudden control changes (e.g., H20).
*   **Suggested Improvement:** Implement more agile inventory controls for products subject to potential export restrictions. This may involve:
    *   Shorter lead times for production of sensitive SKUs.
    *   Enhanced forecasting models that incorporate geopolitical risk scenarios to adjust production volumes before products are ready for market, preventing the "excess inventory" scenario described.

#### C. Strengthen Supplier Resilience and Financial Monitoring
*   **Current Risk:** Insolvency of key suppliers and distributors; inability of suppliers to deliver.
*   **Suggested Improvement:**
    *   Conduct rigorous financial health monitoring of "key suppliers, distributors, and CSPs" to detect insolvency risks early.
    *   Diversify the supplier base to avoid over-reliance on any single entity, ensuring that if one supplier fails to meet "supply commitments," alternative sources can fulfill demand.

#### D. Regulatory Compliance & Legal Strategy Enhancement
*   **Current Risk:** Compliance burdens from shifting export controls; potential penalties from China’s antitrust regulators regarding the Mellanox acquisition.
*   **Suggested Improvement:**
    *   Increase investment in legal and compliance teams to navigate "repeated changes in the export control rules."
    *   Proactively engage with regulators in key markets (including China) to address concerns regarding "degraded products" and compliance with acquisition terms (Mellanox), to avoid "financial penalties" or "restrictions on our ability to conduct our business."

#### E. Product Design for Global Compliance
*   **Current Risk:** Customers "design-out" U.S. semiconductors to reduce compliance burden; foreign competitors benefit from being outside the scope of controls.
*   **Suggested Improvement:**
    *   Accelerate the development of product variants that are compliant with a wider range of international regulations, reducing the incentive for customers to "design-out" Nvidia products.
    *   Focus R&D on technologies that are less likely to be subject to "unilateral export controls," thereby maintaining competitiveness in markets like Europe, Latin America, and Southeast Asia.

### Conclusion

Nvidia’s supply chain risk is currently dominated by **geopolitical export controls** rather than traditional manufacturing defects. The critical vulnerability lies in the **concentration of distribution in Hong Kong** and the **lag between product readiness and regulatory changes** (as seen with the H20). To mitigate this, Nvidia must shift from a static supply chain model to a **geopolitically agile** framework, focusing on diversified logistics, dynamic inventory for sensitive SKUs, and enhanced regulatory compliance to protect its market position against competitors not subject to U.S. controls.

## Sources
- Nvidia.pdf | Page 27
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 34
- Nvidia.pdf | Page 39
- Nvidia.pdf | Page 18
- Nvidia.pdf | Page 24

In [26]:
query = "Discuss about operating expenses(R&D and SG&A) of Nvidia in depth and also discuss about trends"

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Discuss about operating expenses(R&D and SG&A) of Nvidia in ...
   Retrieval Quality: 2/3 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 2/3 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7630 | Recall: 0.8126 | F1: 0.7870


# Financial Analysis — Nvidia

### Financial Analysis: NVIDIA Operating Expenses (R&D and SG&A)

The following analysis details the trends in NVIDIA’s operating expenses, specifically Research and Development (R&D) and Sales, General, and Administrative (SG&A), for the fiscal years ended January 25, 2026, and January 26, 2025.

#### 1. Key Financial Figures

| Metric | FY 2026 (Jan 25, 2026) | FY 2025 (Jan 26, 2025) | Change ($) | % Change |
| :--- | :--- | :--- | :--- | :--- |
| **Research and Development** | $18,497 million | $12,914 million | $5,583 million | 43% |
| **Sales, General and Administrative** | $4,579 million | $3,491 million | $1,088 million | 31% |
| **Total Operating Expenses** | $23,076 million | $16,405 million | $6,671 million | 41% |

*Note: Percentage changes are calculated based on the figures provided in the context. For example, for R&D: ((18,497 - 12,914) / 12,914) × 100 = 43.2%, which aligns with the 43% reported in the context.*

#### 2. Detailed Analysis of Trends

**Research and Development (R&D)**
*   **Absolute Growth:** R&D expenses increased from $12,914 million in FY 2025 to $18,497 million in FY 2026, representing an absolute increase of $5,583 million.
*   **Percentage Growth:** This represents a 43% year-over-year increase.
*   **Relative to Revenue:** As a percentage of revenue, R&D expenses declined from 9.9% in FY 2025 to 8.6% in FY 2026. This indicates that while the company is investing more in absolute terms, the revenue growth (65%) is outpacing the growth in R&D spending, leading to improved operational leverage in this category.

**Sales, General and Administrative (SG&A)**
*   **Absolute Growth:** SG&A expenses increased from $3,491 million in FY 2025 to $4,579 million in FY 2026, representing an absolute increase of $1,088 million.
*   **Percentage Growth:** This represents a 31% year-over-year increase.
*   **Relative to Revenue:** As a percentage of revenue, SG&A expenses declined from 2.7% in FY 2025 to 2.1% in FY 2026. Similar to R&D, the growth in SG&A is significantly lower than the growth in revenue, indicating efficient scaling of administrative and sales functions relative to the top-line expansion.

**Total Operating Expenses**
*   **Aggregate Trend:** Total operating expenses rose from $16,405 million in FY 2025 to $23,076 million in FY 2026, a 41% increase.
*   **Operational Leverage:** The total operating expense ratio (as a percentage of revenue) decreased from 12.6% in FY 2025 to 10.7% in FY 2026. This decline in the expense ratio, despite significant absolute increases in both R&D and SG&A, demonstrates strong operational leverage. The company is successfully scaling its revenue base faster than its operating cost base, contributing to the increase in operating income from $81,453 million in FY 2025 to $130,387 million in FY 2026.

#### 3. Strategic Context
The context notes that the year-over-year increase in Compute & Networking revenue was driven by major platform shifts, specifically accelerated computing and AI, with Data Center computing revenue growing 59% driven by demand for the Blackwell computing platform. The substantial increase in R&D expenses (43%) aligns with this strategic focus, reflecting continued investment in developing these high-growth AI and accelerated computing platforms.

## Sources
- Nvidia.pdf | Page 105
- Nvidia.pdf | Page 71
- Nvidia.pdf | Page 57
- Nvidia.pdf | Page 58
- Nvidia.pdf | Page 104
- Nvidia.pdf | Page 53
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 75

In [27]:
query = """Evaluate NVIDIA's overall corporate strategy and capital allocation decisions.

Focus on:
- Core business model and diversification efforts
- R&D investment trends
- Mergers & acquisitions strategy
- Share buyback and dividend policy
- Long-term vision in AI, robotics, autonomous vehicles, and Omniverse
- Management's capital allocation priorities

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 

Provide investor implications."""

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Evaluate NVIDIA's overall corporate strategy and capital all...
   Retrieval Quality: 11/18 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 11/18 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7832 | Recall: 0.7798 | F1: 0.7815


# Financial Analysis — Nvidia

**Financial Analysis: NVIDIA Corporate Strategy and Capital Allocation**

**Note on Data Availability:**
The provided Context contains qualitative descriptions of NVIDIA’s business segments, strategic investments, and regulatory environment. However, it **does not contain** specific financial figures for Revenue, Operating Expenses, R&D, Net Income, Cash Flow, Share Buybacks, or Dividends for any fiscal year. Consequently, quantitative trend analysis (e.g., percentage changes in R&D or revenue growth rates) is **not available in the retrieved sections**. The following analysis is based strictly on the qualitative strategic disclosures provided in the Context.

### 1. Core Business Model and Diversification Efforts

NVIDIA has transitioned from a company focused on PC graphics to a "data center scale AI infrastructure company reshaping all industries." The core business model relies on accelerated computing, leveraging the NVIDIA CUDA development platform and a deep software stack (libraries, frameworks, SDKs, APIs) to address computationally intensive workloads.

*   **Segment Structure:** The company operates through two distinct segments:
    1.  **Compute & Networking:** Includes Data Center accelerated computing and networking platforms, AI solutions and software, and Automotive platforms (autonomous and electric vehicle solutions).
    2.  **Graphics:** Includes GeForce GPUs for gaming/PCs and Quadro/NVIDIA RTX GPUs for enterprise workstation graphics.
*   **Diversification Strategy:** NVIDIA has expanded beyond its original PC graphics focus into scientific computing, AI, data science, autonomous vehicles, robotics, and digital twin applications. This diversification is driven by the sustained demand for 3D graphics and the scale of the gaming market, which provided the foundational GPU architecture now applied to broader industrial and AI applications.
*   **Strategic Positioning:** The company positions itself as a holistic infrastructure provider. With the Blackwell architecture, NVIDIA emphasizes "extreme co-design," where chips, networking, systems, software, and algorithms are holistically architected to maximize performance and scale, allowing hundreds of thousands of GPUs to function as a single giant computer.

### 2. R&D Investment Trends

The Context does not disclose specific Research & Development (R&D) dollar amounts or trends. However, it highlights the strategic importance of R&D through the following points:
*   **Technology Roadmap:** NVIDIA states it has made and expects to continue making investments that support its technology roadmap and the broader AI ecosystem.
*   **Architecture Evolution:** The introduction of the Blackwell architecture represents a significant R&D output, featuring data-center-scale offerings with extreme co-design.
*   **Software Stack Depth:** The development of hundreds of domain-specific software libraries, frameworks, and APIs indicates a continuous investment in software R&D to facilitate deployment across verticals such as healthcare, telecom, automotive, and manufacturing.

### 3. Mergers & Acquisitions Strategy

The Context does not disclose specific recent M&A transactions or a detailed M&A strategy. However, it provides insight into the company's approach to acquisitions and investments:
*   **Accounting Treatment:** NVIDIA applies a screen test to determine if a transaction is an asset acquisition or a business combination. For business combinations, the fair value of the purchase price is allocated to tangible assets, liabilities, and intangible assets, with the excess recorded as goodwill.
*   **Measurement Period:** Adjustments to assets and liabilities are recorded during a measurement period of up to one year from the acquisition date, with subsequent adjustments impacting the Consolidated Statements of Income.
*   **Strategic Investments:** While not strictly M&A, NVIDIA has made significant strategic investments in private companies and infrastructure funds. In fiscal year 2026, NVIDIA invested **$17.5 billion** in private companies and infrastructure funds, primarily to support early-stage startups, including AI model makers that purchase NVIDIA products directly or through Cloud Service Providers (CSPs). These investments are described as illiquid and non-marketable, with no assurance of return.

### 4. Share Buyback and Dividend Policy

**Not disclosed in the retrieved sections.** The provided Context does not contain any information regarding NVIDIA’s share repurchase programs, dividend payments, or capital return policies to shareholders.

### 5. Long-term Vision in AI, Robotics, Autonomous Vehicles, and Omniverse

NVIDIA’s long-term vision is centered on becoming a data center scale AI infrastructure company that reshapes all industries. Key elements of this vision include:

*   **AI Infrastructure:** The company focuses on providing the foundational infrastructure for AI model training and inference. The Blackwell architecture is central to this, enabling massive scale-out computing.
*   **Robotics and Autonomous Vehicles:** NVIDIA has expanded into autonomous vehicles and robotics, offering platforms and software solutions. The Compute & Networking segment specifically includes Automotive platforms for autonomous and electric vehicles.
*   **Digital Twins and Omniverse:** While "Omniverse" is not explicitly named in the provided text, the Context mentions "digital twin applications" as one of the fields NVIDIA has expanded into. This aligns with the company’s strategy to leverage GPU architecture for simulation and digital representation of physical systems.
*   **Industry Verticals:** The company targets vertical-specific optimizations for industries ranging from healthcare and telecom to automotive and manufacturing, indicating a broad-based long-term vision beyond just data centers.

### 6. Management's Capital Allocation Priorities

Management’s capital allocation priorities, as inferred from the Context, focus on:
*   **Strategic Ecosystem Investment:** A significant portion of capital is allocated to supporting the broader AI ecosystem. This includes the **$17.5 billion** investment in private companies and infrastructure funds in fiscal year 2026.
*   **Infrastructure Guarantees:** NVIDIA has provided **$3.5 billion** in land, power, and shell guarantees to early-stage companies to support the build-out of complex data center infrastructures. These guarantees are generally over multi-year periods.
*   **Technology Roadmap:** Continued investment in the technology roadmap to maintain leadership in accelerated computing and AI infrastructure.
*   **Regulatory Compliance:** Allocation of resources to manage increased compliance costs and regulatory scrutiny from global authorities (EU, US, UK, South Korea, Japan, China).

### Investor Implications and Advice

Based on the qualitative analysis of NVIDIA’s strategy and capital allocation:

1.  **Ecosystem Dependency Risk:** NVIDIA’s strategy of investing **$17.5 billion** in private companies and providing **$3.5 billion** in infrastructure guarantees indicates a deep integration into the AI ecosystem. Investors should be aware that the success of these investments is tied to the profitability of early-stage startups, which may not become profitable in the near term. The illiquid nature of these investments poses a risk if the AI market slows down.
2.  **Regulatory and Geopolitical Risk:** The Context highlights increased interest from regulators worldwide, including the European Union, United States, United Kingdom, South Korea, Japan, and China. NVIDIA faces potential compliance costs and operational restrictions due to antitrust legislation and cybersecurity regulations. Investors should monitor regulatory developments, particularly in China and the EU, as they could impact revenue from key markets.
3.  **Infrastructure Constraints:** The availability of data centers, energy, and capital is crucial for NVIDIA’s growth. Any shortage of these resources could impact future revenue. Investors should consider the macroeconomic factors, such as energy capacity expansion challenges and capital market volatility, as potential headwinds.
4.  **Strategic Focus on Data Centers:** With Blackwell architectures representing the majority of Data Center revenue, investors should focus on the company’s ability to maintain its lead in data center AI infrastructure. The "extreme co-design" approach is a competitive advantage, but it also requires continuous innovation.
5.  **Lack of Capital Return Information:** The absence of information on share buybacks and dividends in the provided Context means investors cannot assess NVIDIA’s capital return policy from this document. Investors should refer to other sections of the 10-K or earnings calls for details on how NVIDIA returns capital to shareholders.

**Recommendation for Investors:**
*   **Monitor Regulatory Developments:** Keep a close eye on regulatory actions in the US, EU, and China, as these could significantly impact NVIDIA’s market access and compliance costs.
*   **Assess Ecosystem Health:** Evaluate the health of the AI startup ecosystem and the ability of NVIDIA’s investees to generate returns. The **$17.5 billion** investment is a significant commitment, and its success will depend on the broader AI market’s growth.
*   **Watch Infrastructure Trends:** Track developments in data center energy capacity and capital availability, as these are critical for NVIDIA’s future revenue growth.
*   **Review Full Financial Statements:** Since the provided Context lacks detailed financial figures, investors should review the full Consolidated Financial Statements to assess revenue growth, margins, and cash flow trends before making investment decisions.

## Sources
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 81
- Nvidia.pdf | Page 104
- Nvidia.pdf | Page 4
- Nvidia.pdf | Page 83
- Nvidia.pdf | Page 53

In [28]:
query = """Provide a comprehensive financial analysis of Tesla using the latest SEC 10-k filings.

Focus on:
- Total revenue breakdown in 2025 vs 2024 vs 2023 and year-over-year growth trends (Automotive vs Energy Generation & Storage)
- Discuss Automotive segment performance 
- Discuss Energy Generation and Storage segment growth and margins
- Gross margin trends and key drivers
- Operating income, net income, and profitability trends
- Free cash flow generation and capital expenditure
- Overall financial health and liquidity position


Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors."""


response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Provide a comprehensive financial analysis of Tesla using th...
   Retrieval Quality: 37/24 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 37/24 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7372 | Recall: 0.7384 | F1: 0.7378


# Financial Analysis — Tesla

# Financial Analysis: Tesla, Inc. (FY 2023–FY 2025)

## 1. Revenue Breakdown and Growth Trends

Tesla’s total revenue has experienced a slight contraction in the most recent period, driven by a significant decline in the core automotive segment, partially offset by robust growth in energy and services.

| Metric | FY 2023 | FY 2024 | FY 2025 | Change (2025 vs 2024) | % Change (2025 vs 2024) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Total Revenues** | $96,773 million | $97,690 million | $94,827 million | -$2,863 million | -2.9% |
| **Total Automotive Revenues** | $82,419 million | $77,070 million | $69,526 million | -$7,544 million | -9.8% |
| **Energy Generation and Storage** | $6,035 million | $10,086 million | $12,771 million | +$2,685 million | +26.6% |
| **Services and Other** | $8,319 million | $10,534 million | $12,530 million | +$1,996 million | +18.9% |

*Calculations:*
*   Total Revenue % Change: $((94,827 - 97,690) / 97,690) \times 100 = -2.9\%$
*   Automotive Revenue % Change: $((69,526 - 77,070) / 77,070) \times 100 = -9.8\%$
*   Energy Generation & Storage % Change: $((12,771 - 10,086) / 10,086) \times 100 = +26.6\%$
*   Services and Other % Change: $((12,530 - 10,534) / 10,534) \times 100 = +18.9\%$

### Automotive Segment Performance
The Automotive segment remains the largest revenue driver but is under pressure. Total automotive revenues decreased from $77,070 million in FY 2024 to $69,526 million in FY 2025. This decline is attributed to a decrease in deliveries year-over-year. Within this segment, Automotive sales revenue fell from $72,480 million in FY 2024 to $65,821 million in FY 2025. Additionally, Automotive regulatory credits revenue declined from $2,763 million in FY 2024 to $1,993 million in FY 2025, and Automotive leasing revenue decreased from $1,827 million in FY 2024 to $1,712 million in FY 2025. The contraction in both unit sales and regulatory credit income indicates a challenging environment for the core vehicle business.

### Energy Generation and Storage Segment Growth
In contrast to the automotive segment, the Energy Generation and Storage segment demonstrated strong expansion. Revenue increased from $10,086 million in FY 2024 to $12,771 million in FY 2025. The context notes this increase was primarily due to increases in Megapack and Powerwall deployments, partially offset by a decrease in the average selling price of Megapack. This segment is becoming an increasingly material contributor to the company's top line, growing at a significantly higher rate than the overall company revenue.

## 2. Gross Margin Trends and Key Drivers

Gross profit has shown a consistent downward trend over the three-year period, reflecting margin compression across the business.

| Metric | FY 2023 | FY 2024 | FY 2025 | Change (2025 vs 2024) |
| :--- | :--- | :--- | :--- | :--- |
| **Gross Profit** | $17,660 million | $17,450 million | $17,094 million | -$356 million |
| **Total Cost of Revenues** | $79,113 million | $80,240 million | $77,733 million | -$2,507 million |

*   **Gross Margin Calculation (FY 2025):** $17,094 / 94,827 = 18.0\%$
*   **Gross Margin Calculation (FY 2024):** $17,450 / 97,690 = 17.9\%$
*   **Gross Margin Calculation (FY 2023):** $17,660 / 96,773 = 18.2\%$

**Key Drivers:**
*   **Automotive Gross Margin:** Decreased from 18.4% in FY 2024 to 17.8% in FY 2025. The context attributes this primarily to a decrease in regulatory credits revenue and changes in automotive sales revenue and cost of automotive sales revenue.
*   **Automotive & Services Gross Margin:** Decreased from 16.9% in FY 2024 to 16.2% in FY 2025. This was driven by the decrease in regulatory credits, partially offset by an improvement in services and other margins.
*   **Cost Dynamics:** Cost of automotive sales revenue decreased by $5.60 billion (9%) in FY 2025 compared to FY 2024, due to lower deliveries and lower average cost per unit (sales mix and lower material costs). However, this was partially offset by lower fixed cost absorption and an increase in tariffs. Conversely, Cost of services and other revenue increased by $1.68 billion (17%) in FY 2025, driven by higher volumes in used vehicle sales, Supercharging, insurance, and maintenance.

## 3. Operating Income, Net Income, and Profitability Trends

Profitability has declined significantly year-over-year, with Operating Income and Net Income both showing substantial drops.

| Metric | FY 2023 | FY 2024 | FY 2025 | Change (2025 vs 2024) | % Change (2025 vs 2024) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Income from Operations** | $8,891 million | $7,076 million | $4,355 million | -$2,721 million | -38.5% |
| **Net Income** | $14,974 million | $7,153 million | $3,855 million | -$3,298 million | -46.1% |
| **Net Income Attributable to Common Stockholders** | $14,997 million | $7,091 million | $3,794 million | -$3,297 million | -46.5% |
| **Diluted EPS** | $4.30 | $2.04 | $1.08 | -$0.96 | -47.1% |

*Calculations:*
*   Operating Income % Change: $((4,355 - 7,076) / 7,076) \times 100 = -38.5\%$
*   Net Income % Change: $((3,855 - 7,153) / 7,153) \times 100 = -46.1\%$

**Analysis:**
*   **Operating Expenses:** Total operating expenses increased from $10,374 million in FY 2024 to $12,739 million in FY 2025. This increase was driven by Research and Development expenses rising from $4,540 million to $6,411 million and Selling, general and administrative expenses rising from $5,150 million to $5,834 million. Restructuring and other expenses decreased from $684 million to $494 million.
*   **Tax Impact:** The Provision for income taxes was $1,423 million in FY 2025, compared to $1,837 million in FY 2024.
*   **Non-Operating Items:** Interest income increased from $1,569 million in FY 2024 to $1,680 million in FY 2025. However, "Other (expense) income, net" swung from a positive $695 million in FY 2024 to a negative $(419) million in FY 2025, further pressuring the bottom line.

## 4. Free Cash Flow Generation and Capital Expenditure

The provided context does not contain specific line items for "Capital Expenditure" or "Free Cash Flow." However, it provides data on Cash Flows from Financing Activities and total cash positions.

*   **Net Cash Flows from Financing Activities:** Decreased by $2.71 billion to $1.14 billion during the year ended December 31, 2025, from $3.85 billion during the year ended December 31, 2024. This decrease was primarily due to a $3.05 billion increase in repayments of debt and a $158 million decrease in proceeds from issuances of debt.
*   **Note:** Specific figures for Operating Cash Flow and Capital Expenditures are not disclosed in the retrieved sections. Therefore, a precise calculation of Free Cash Flow is not possible based solely on the provided text.

## 5. Overall Financial Health and Liquidity Position

Tesla maintains a strong liquidity position, with total cash and cash equivalents increasing year-over-year.

| Metric | Dec 31, 2023 | Dec 31, 2024 | Dec 31, 2025 |
| :--- | :--- | :--- | :--- |
| **Cash and cash equivalents** | $16,398 million | $16,139 million | $16,513 million |
| **Restricted cash (Current)** | $543 million | $494 million | $389 million |
| **Restricted cash (Non-current)** | $248 million | $404 million | $714 million |
| **Total Cash and Restricted Cash** | $17,189 million | $17,037 million | $17,616 million |

**Analysis:**
*   **Liquidity:** Total cash and restricted cash increased

## Sources
- Tesla.pdf | Page 54
- Tesla.pdf | Page 66
- Tesla.pdf | Page 64
- Tesla.pdf | Page 77
- Tesla.pdf | Page 74
- Tesla.pdf | Page 99
- Tesla.pdf | Page 82

In [29]:
query = "Discuss about operating expenses of Tesla in depth"

"Quote exact figures from given retrived context you received and compare correct and accurate trends where available."

response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Discuss about operating expenses of Tesla in depth
   Retrieval Quality: 3/2 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 3/2 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7443 | Recall: 0.8411 | F1: 0.7897


# Financial Analysis — Tesla

### Financial Analysis: Tesla Operating Expenses

Based on the retrieved financial statements and management discussion, the following analysis details Tesla’s operating expenses for the fiscal years ended December 31, 2023, 2024, and 2025.

#### 1. Key Operating Expense Figures

| Metric | FY 2023 | FY 2024 | FY 2025 | Change (2025 vs 2024) | % Change (2025 vs 2024) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Research and development** | $3,969 million | $4,540 million | $6,411 million | $1,871 million | 41.2% |
| **Selling, general and administrative** | $4,800 million | $5,150 million | $5,834 million | $684 million | 13.3% |
| **Restructuring and other** | $0 million | $684 million | $494 million | ($190) million | -27.8% |
| **Total operating expenses** | $8,769 million | $10,374 million | $12,739 million | $2,365 million | 22.8% |

*Note: Percentages calculated as ((Later Year - Earlier Year) / Earlier Year) × 100.*

#### 2. Detailed Component Analysis

**Research and Development (R&D)**
*   **Trend:** R&D expense increased significantly, rising from $4,540 million in 2024 to $6,411 million in 2025.
*   **Drivers:** The increase of $1,871 million (41%) was primarily driven by higher costs related to AI and other programs as Tesla expanded its product roadmap and technologies. Additionally, there was a $500 million increase in stock-based compensation.
*   **Relative Impact:** R&D expense as a percentage of revenue increased from 5% in 2024 to 7% in 2025. This rise is attributed to the absolute increase in R&D spending combined with a decrease in total revenue year-over-year.

**Selling, General and Administrative (SG&A)**
*   **Trend:** SG&A expense increased from $5,150 million in 2024 to $5,834 million in 2025.
*   **Drivers:** The $684 million (13%) increase was driven by:
    *   A $354 million increase in operating expenses, including legal charges.
    *   A $256 million increase in employee and labor costs, including professional services.
    *   A $235 million increase in stock-based compensation.
    *   These increases were partially offset by an $83 million decrease in marketing expenses and a $78 million decrease in facilities-related expenses.
*   **Relative Impact:** SG&A expense as a percentage of revenues increased from 5% in 2024 to 6% in 2025.

**Restructuring and Other**
*   **Trend:** This line item decreased from $684 million in 2024 to $494 million in 2025.
*   **Context:** In 2023, this expense was not disclosed (represented as "—" in the table, implying $0 or immaterial/not applicable for that specific line item in the provided text). The decrease in 2025 reflects lower restructuring costs compared to the prior year.

**Total Operating Expenses**
*   **Trend:** Total operating expenses rose from $10,374 million in 2024 to $12,739 million in 2025, representing a $2,365 million increase.
*   **Impact on Operating Income:** This increase in operating expenses, coupled with a decrease in Gross Profit (from $17,450 million in 2024 to $17,094 million in 2025), resulted in a significant decline in Income from operations. Income from operations decreased from $7,076 million in 2024 to $4,355 million in 2025.

#### 3. Strategic Context
The substantial increase in R&D expenses aligns with Tesla’s stated strategy of investing in AI initiatives, compute infrastructure, and data centers. Management notes that capital expenditures are expected to exceed $20 billion in 2026, further driven by AI initiatives and the expansion of manufacturing and R&D production lines. This indicates a continued shift in resource allocation toward technology and AI capabilities, which is currently pressuring operating margins.

## Sources
- Tesla.pdf | Page 120
- Tesla.pdf | Page 82
- Tesla.pdf | Page 66
- Tesla.pdf | Page 68
- Tesla.pdf | Page 72

In [30]:
query = """Analyze Tesla's autonomous driving technology, Full Self-Driving (FSD), and Robotaxi strategy in detail.

Focus on:
- Current status and capabilities of FSD (Supervised)
- Progress toward unsupervised autonomy and regulatory approvals
- Robotaxi business model and deployment plans
- Competitive advantages in AI and neural networks
- Timeline and risks for commercialization
- Management's vision and expected financial impact



Explain each point above in very much depth and with intuition

Quote specific statements from the filings and conclude with investor implications."""

response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Analyze Tesla's autonomous driving technology, Full Self-Dri...
   Retrieval Quality: 16/19 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 16/19 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7858 | Recall: 0.7783 | F1: 0.7820


# Financial Analysis — Tesla

**Financial Analysis: Tesla Autonomous Driving & Robotaxi Strategy**

**Classification:** Type B (Strategy, Technology, and Non-Numeric Topic)
*Note: As the retrieved context contains no financial figures (Revenue, Operating Income, etc.), a financial table is not applicable. The analysis below is derived strictly from the qualitative disclosures provided.*

### 1. Current Status and Capabilities of FSD (Supervised)
Tesla’s Full Self-Driving (FSD) technology is currently deployed as a supervised feature, integrated into its vehicle portfolio. The filing clarifies that the term “FSD (Supervised)” is used globally, with the equivalent naming convention “FSD (Capability)” applied in the European, Middle East, and Asia-Pacific regions.

The company emphasizes that it is “continuing to develop full self-driving technology for improved safety” and is focused on “further improving and deploying our FSD (Supervised) capabilities.” The technology is described as leveraging “neural network capabilities” and is a core component of the company’s strategy to bring “artificial intelligence (“AI”) into the real world.”

> **Quote:** “We are focused on bringing artificial intelligence (“AI”) into the real world, through products and services like Full Self-Driving (“FSD”) (Supervised) and Robotaxi... We emphasize performance, attractive styling and the safety of our users and workforce in the design and manufacture of our products and are continuing to develop full self-driving technology for improved safety.”

### 2. Progress Toward Unsupervised Autonomy and Regulatory Approvals
The provided text does not explicitly detail specific regulatory approval milestones or a confirmed timeline for unsupervised (Level 4/5) autonomy. However, it confirms the transition from supervised to autonomous operations through the launch of the Robotaxi service.

The filing notes that Tesla is working to “develop and commercialize AI robots (“Bots”) (including Optimus)” and is leveraging “proprietary Full Self-Driving (“FSD”) (Supervised) features, battery cell and other technologies.” The shift toward unsupervised autonomy is evidenced by the launch of the Robotaxi service, which is described as an “autonomous ride-hailing platform.”

> **Quote:** “In June 2025, we launched our Robotaxi service, an autonomous ride-hailing platform that harnesses our technology and vehicles.”

### 3. Robotaxi Business Model and Deployment Plans
Tesla launched its Robotaxi service in **June 2025**. The business model is designed to be “service-driven,” based on “AI, software and fleet-based profits.”

*   **Current Fleet:** The service currently operates using **Model Y** vehicles.
*   **Future Fleet:** The company plans to include **Cybercab**, described as a “purpose-built autonomous vehicle,” in the future.
*   **Deployment Strategy:** Tesla states it has “continued to expand and refine our Robotaxi service after its June 2025 launch, capitalizing on our AI investments and scalable mobility infrastructure.”
*   **Objective:** The service aims to “open access to an expanded customer base as modes of transportation evolve.”

> **Quote:** “Our Robotaxi business currently operates with Model Y vehicles but, in time, will include Cybercab, our purpose-built autonomous vehicle.”

### 4. Competitive Advantages in AI and Neural Networks
Tesla positions its competitive advantage in the “fields of AI and robotics.” The company intends to compete in the developing autonomous market by leveraging:
*   Continued progress on “FSD (Supervised) and neural network capabilities.”
*   Its “Supercharger network and infotainment offerings.”
*   A strategy to “vertically integrate and localize our supply chain” and “expand our global infrastructure, including our service and charging infrastructure.”

The filing highlights that Tesla is focused on “profitable growth via a differentiated and efficiently managed product portfolio” that leverages existing factories and production lines.

> **Quote:** “We expect our Robotaxi service to compete in this developing market, along with traditional ride-hailing and taxi services, through continued progress on our FSD (Supervised) and neural network capabilities, Supercharger network and infotainment offerings.”

### 5. Timeline and Risks for Commercialization
**Timeline:**
*   **June 2025:** Launch of Robotaxi service.
*   **Future:** Mass production of Cybercab and inclusion in the Robotaxi fleet.
*   **2025:** Production of approximately 1.66 million consumer vehicles and delivery of approximately 1.64 million consumer vehicles.

**Key Risks:**
*   **Adoption Risk:** Success is “dependent upon various factors, including the acceptance and adoption by consumers of autonomous driving solutions, and Robotaxi as a preferable option, amid growing competition.”
*   **Market Conditions:** If “unfavorable global market conditions” occur or if the company cannot maintain operations at a commensurate scope, “our business, prospects, financial condition and operating results may be harmed.”
*   **Competitive Pressure:** Tesla faces “strong competition for our products and services from a growing list of established and new competitors.”
*   **Legal Risk:** A proposed class action was filed on **August 4, 2025**, alleging violations of federal securities laws regarding “material misrepresentations in public filings regarding the effectiveness of Autopilot, Full-Self Driving (Supervised), and Robotaxi.” The company states it “cannot predict the outcome or impact.”

> **Quote:** “If the uptake rate for autonomous driving solutions does not develop as we expect, our business, prospects, financial condition and operating results may be harmed.”

### 6. Management's Vision and Expected Financial Impact
Management’s vision is to transition Tesla into a top provider of autonomous solutions and AI robots. The strategic goal is to “unlock the potential to advance a service-driven business model based on AI, software and fleet-based profits.”

The company intends to leverage its current operations in electric vehicles and energy storage to achieve this objective. The focus is on “profitable growth” by reducing manufacturing costs, increasing vehicle production, and improving AI compute technologies. The Robotaxi service is expected to “open access to an expanded customer base,” thereby diversifying revenue streams beyond vehicle sales.

> **Quote:** “We expect this service will open access to an expanded customer base as modes of transportation evolve, and, along with products such as FSD (Supervised) subscriptions, unlock the potential to advance a service-driven business model based on AI, software and fleet-based profits.”

### Investor Implications

1.  **Shift to Service Revenue:** The launch of Robotaxi in June 2025 marks a pivotal shift from a pure hardware manufacturer to a service-driven entity. Investors should monitor the scalability of the Model Y-based Robotaxi fleet and the timeline for Cybercab mass production as key drivers for future software and fleet-based profits.
2.  **Regulatory and Legal Exposure:** The August 2025 class action lawsuit regarding the effectiveness of FSD and Robotaxi represents a significant legal risk. While Tesla intends to “vigorously defend itself,” the potential for monetary damages and reputational impact could affect financial condition and operating results.
3.  **Competitive Intensity:** The autonomous market is described as “highly competitive” with both established and emerging players. Tesla’s reliance on “neural network capabilities” and its “Supercharger network” are cited as competitive differentiators, but the success of the Robotaxi model is contingent on consumer adoption rates, which remain uncertain.
4.  **Operational Scale:** With 2025 production at approximately 1.66 million vehicles, Tesla’s ability to “vertically integrate and localize our supply chain” will be critical in maintaining margins as it scales its autonomous fleet and energy storage deployments (46.7 GWh in 2025).
5.  **Technology Integration:** The integration of FSD (Supervised) with the Robotaxi platform and future AI robots (Optimus) suggests a long-term strategy where AI capabilities are the core value driver. Investors should view Tesla’s valuation through the lens of its AI and robotics progress rather than solely its automotive sales volume.

## Sources
- Tesla.pdf | Page 6
- Tesla.pdf | Page 140
- Tesla.pdf | Page 20
- Tesla.pdf | Page 54
- Tesla.pdf | Page 28

In [31]:
query = """Provide a detailed analysis of Tesla's Energy Generation and Storage segment.

Focus on:
- Powerwall and Megapack products and their applications
- Revenue growth, deployment volumes, and profitability trends
- Virtual power plants and software platforms (Powerhub, Autobidder)
- Competitive positioning vs traditional utilities and other energy storage companies
- Future growth outlook and strategic importance to Tesla

Explain each point above in very much depth and with intuition

Use exact figures and management commentary from the filings."""

response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Provide a detailed analysis of Tesla's Energy Generation and...
   Retrieval Quality: 23/18 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 23/18 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7660 | Recall: 0.8373 | F1: 0.8000


# Financial Analysis — Tesla

**Financial Analysis: Tesla Energy Generation and Storage Segment**

### 1. Product Architecture and Application Scope

Tesla’s Energy Generation and Storage segment is bifurcated into two primary product lines: **Energy Storage Products** and **Energy Generation Systems**. The segment leverages component-level technologies derived from Tesla’s automotive division to optimize manufacturing capacity and grid integration.

*   **Megapack:**
    *   **Application:** Designed for commercial, industrial, utility, and energy generation customers.
    *   **Scale:** Multiple units can be grouped to form installations of gigawatt hours (“GWh”) or greater capacity.
    *   **Strategic Role:** As AI infrastructure drives rapid load growth, Megapack is positioned to increase the utilization of existing generation and transmission capacity, thereby enhancing the efficiency of the electric grid.
*   **Powerwall:**
    *   **Application:** Designed to store energy at homes or small commercial facilities.
    *   **Distribution:** Sold and leased directly to customers as well as through channel partners.
*   **Energy Generation Systems:**
    *   **Products:** Includes solar panels and **Solar Roof**, which combines premium glass roof tiles with energy generation.
    *   **Manufacturing:** Tesla designs and manufactures certain components, including solar panels. A new residential retrofit solar panel began manufacturing in 2025, with initial customer deliveries commencing in January 2026.
    *   **Integration:** Both generation products are designed to integrate with Powerwall. Efficiency is aided by Tesla’s proprietary solar inverter, which incorporates the company’s power electronics technologies.

### 2. Financial Performance: Revenue, Costs, and Profitability

The segment demonstrated significant top-line growth and margin expansion in the year ended December 31, 2025, compared to the year ended December 31, 2024.

| Metric | Year Ended Dec 31, 2024 | Year Ended Dec 31, 2025 | Change ($ Billion) | % Change |
| :--- | :--- | :--- | :--- | :--- |
| **Energy Generation and Storage Revenue** | Not Disclosed in Retrieved Sections | Not Disclosed in Retrieved Sections | $2.69 billion | 27% |
| **Cost of Energy Generation and Storage Revenue** | Not Disclosed in Retrieved Sections | Not Disclosed in Retrieved Sections | $1.52 billion | 20% |
| **Gross Margin** | 26.2% | 29.8% | 3.6 percentage points | N/A |

*Note: The absolute revenue and cost figures for the base year (2024) and the later year (2025) are not explicitly stated as total dollar amounts in the provided text, only the year-over-year changes and percentages. Therefore, the "Earlier Year" and "Later Year" columns for absolute values are marked as not disclosed in the retrieved sections, while the Change columns reflect the verbatim figures provided.*

**Revenue Analysis:**
*   **Growth Driver:** Energy generation and storage revenue increased **$2.69 billion**, or **27%**, in the year ended December 31, 2025, as compared to the year ended December 31, 2024.
*   **Primary Factors:** This increase was primarily driven by increases in **Megapack** and **Powerwall** deployments.
*   **Offsetting Factor:** The volume growth was partially offset by a **decrease in average selling price** of Megapack.

**Cost and Margin Analysis:**
*   **Cost Increase:** Cost of energy generation and storage revenue increased **$1.52 billion**, or **20%**, in the year ended December 31, 2025, as compared to the year ended December 31, 2024.
*   **Cost Drivers:** The increase was primarily from increases in Megapack and Powerwall deployments.
*   **Cost Mitigation:** This was partially offset by a **decrease in average cost per unit** for Megapack and Powerwall. This reduction was attributed to:
    *   Lower raw material costs.
    *   Lower manufacturing costs for Megapack, in part from the ramp of the **Shanghai Megafactory**.
    *   *Offsetting Cost Pressure:* Higher tariffs partially offset these savings.
*   **Margin Expansion:** Gross margin for the segment increased from **26.2%** in 2024 to **29.8%** in 2025. This expansion was primarily due to the changes in revenue and cost of revenue discussed above, indicating that revenue growth outpaced cost growth.

### 3. Software Platforms and Virtual Power Plants

Tesla is leveraging its AI capabilities to enhance the value proposition of its hardware through software-defined optimization.

*   **Autobidder:** A real-time energy control and optimization platform specifically for **Megapack** batteries. It enables remote control and dispatch of these large-scale storage systems.
*   **Powerhub:** A platform for **distributed energy resources (DERs)**, including **Powerwall**-enabled **virtual power plants**. It allows for the aggregation and optimization of smaller, distributed storage units.
*   **Strategic Intuition:** By integrating AI-driven firmware updates and software platforms, Tesla transforms static hardware into dynamic grid assets. This allows for "fast-acting systems for power injection and absorption," enabling the battery systems to interconnect with electricity grids more efficiently. This software layer creates a recurring value proposition beyond the initial hardware sale, potentially increasing customer stickiness and enabling new revenue streams through grid services.

### 4. Competitive Positioning and Strategic Importance

*   **Grid Efficiency vs. Traditional Utilities:** The filing explicitly states that as AI infrastructure drives rapid load growth, Megapack helps to "increase utilization of existing generation and transmission capacity, resulting in a more efficient use of the electric grid." This positions Tesla not just as a hardware vendor, but as a critical enabler of grid stability and efficiency, competing indirectly with traditional utility expansion projects by optimizing existing infrastructure.
*   **Manufacturing Scale:** The ramp of the **Shanghai Megafactory** is cited as a key driver for lower manufacturing costs. This suggests a competitive advantage in unit economics, allowing Tesla to maintain or improve margins even as average selling prices for Megapack decrease.
*   **Segment Structure:** The Energy Generation and Storage segment is one of only two reportable segments (alongside Automotive), managed by the CEO as the Chief Operating Decision Maker (CODM). This highlights the strategic importance of the energy business to Tesla’s overall corporate structure and resource allocation.

### 5. Future Growth Outlook

*   **AI-Driven Demand:** The context identifies **AI infrastructure** as a driver of "rapid load growth." This implies a sustained demand tailwind for Megapack, as data centers and AI workloads require significant and stable power supply, which energy storage can provide.
*   **Product Pipeline:** The launch of a **new residential retrofit solar panel** (manufacturing started 2025, deliveries started January 2026) signals continued innovation in the generation side of the business, aiming to capture the residential market with more accessible retrofit options alongside the premium Solar Roof.
*   **Margin Trajectory:** The expansion of gross margin from **26.2%** to **29.8%** suggests that the segment is moving up the value chain, with software and scale efficiencies contributing to higher profitability. If the trend of cost reductions (via Shanghai Megafactory and lower raw material costs) continues to outpace price decreases, further margin expansion is plausible.

**Conclusion:**
Tesla’s Energy Generation and Storage segment is experiencing robust growth (27% revenue increase) and improving profitability (gross margin up to 29.8%). The segment is strategically positioned to benefit from AI-driven load growth, leveraging a combination of hardware scale (Shanghai Megafactory) and software differentiation (Autobidder, Powerhub) to optimize grid efficiency and capture value from both utility-scale and distributed energy resources.

## Sources
- Tesla.pdf | Page 7
- Tesla.pdf | Page 64
- Tesla.pdf | Page 9
- Tesla.pdf | Page 145
- Tesla.pdf | Page 66

In [32]:
query = """Analyze Tesla's supply chain, manufacturing operations, and associated risks.

Focus on:
- Dependency on key suppliers (especially battery cells)
- Manufacturing facilities and production ramp plans
- Geopolitical and raw material risks (lithium, nickel, etc.)
- Vertical integration efforts and in-house battery cell production
- Impact of potential disruptions on production and financials

Explain each point above in very much depth and with intuition


Assess overall supply chain vulnerability and mitigation strategies."""

response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))




🔍 Company: Tesla | Query: Analyze Tesla's supply chain, manufacturing operations, and ...
   Retrieval Quality: 19/17 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 19/17 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.8028 | Recall: 0.8124 | F1: 0.8075


# Financial Analysis — Tesla

**Financial Analysis: Tesla Supply Chain, Manufacturing Operations, and Risk Assessment**

Based on the retrieved sections from Tesla’s financial documents, the following is a structured analysis of the company’s supply chain architecture, manufacturing footprint, and associated operational risks.

### 1. Manufacturing Footprint and Production Strategy

Tesla maintains a global manufacturing network designed to balance local cost-competitiveness with global scale.

*   **U.S. Facilities:** Operations are located in California, New York, Texas, and Nevada. These sites handle the manufacturing and assembly of vehicles, battery packs, battery cells, energy storage systems, and solar products.
*   **International Facilities:** Manufacturing is also conducted in China and Germany. The strategic rationale for these locations is to increase vehicle affordability for local customers by:
    *   Reducing transportation costs.
    *   Reducing manufacturing costs.
    *   Limiting the impact of unfavorable tariffs.
*   **Capacity Expansion:** The company continues to expand production capacity at existing facilities. A key strategic objective is to increase cost-competitiveness in significant markets by strategically adding local manufacturing, often through partnerships with suppliers.

### 2. Supplier Dependency and Battery Cell Sourcing

Tesla’s supply chain relies on parts sourced from thousands of suppliers globally. However, there is a critical dependency on a limited number of key partners for high-value components.

*   **Key Suppliers:** The company has developed close relationships with key partners supplying battery cells, electronics, and complex vehicle assemblies.
*   **Battery Cell Dependency:** Tesla is currently dependent on the continued supply of lithium-ion battery cells for vehicles and energy storage products.
    *   **Primary Suppliers:** The company relies on **Panasonic** and **Contemporary Amperex Technology Co. Limited (CATL)**.
    *   **Limited Flexibility:** Tesla has fully qualified only a "very limited number" of such suppliers and has limited flexibility in changing suppliers.
*   **Economies of Scale:** Certain components are shared or similar across many product lines, allowing Tesla to leverage pricing efficiencies through economies of scale.
*   **Single-Source Risk:** Similar to other OEMs, some procured components and systems are sourced from single suppliers.

### 3. Vertical Integration and In-House Production

To mitigate external dependency and reduce costs, Tesla is pursuing vertical integration, particularly in battery technology.

*   **In-House Battery Cell Production:** In the long term, Tesla intends to supplement supplier-provided cells with cells manufactured in-house. The company believes these in-house cells will be:
    *   More efficient.
    *   Manufacturable at greater volumes.
    *   More cost-effective than currently available cells.
*   **Investment Requirements:** Developing and manufacturing these cells requires significant investments. There is no assurance that Tesla will achieve these targets within planned timeframes.
*   **Lithium Refinery:** As part of vertical integration efforts, Tesla operates an in-house lithium refinery in Texas, which began operations in **January 2026**.

### 4. Raw Material and Geopolitical Risks

The supply chain is exposed to volatility in raw material prices and geopolitical shifts.

*   **Raw Materials:** Products use aluminum, steel, lithium, nickel, and copper.
    *   **Price Volatility:** Pricing is governed by market conditions and may fluctuate due to supply/demand dynamics and market speculation.
    *   **Supply Constraints:** Increased global production of electric vehicles and energy storage products may result in suppliers being unable to meet Tesla’s volume needs.
    *   **Cost Impact:** Increases in raw material prices may reduce profitability if Tesla cannot recoup costs through increased product prices.
*   **Geopolitical and Trade Risks:**
    *   **Tariffs:** U.S. trade policy alterations in **2025**, including heightened import tariffs and retaliatory measures, have impacted supply chain costs.
    *   **Export Controls:** The availability of certain technologies or components may be impacted by retaliatory export controls.
    *   **Mitigation:** Tesla strives to execute long-term supply contracts at competitive pricing and actively manages raw material supply.

### 5. Supply Chain Vulnerability and Mitigation Strategies

Tesla faces multiple potential sources of component shortages due to the global nature of its parts sourcing (thousands of parts from hundreds of suppliers).

*   **Risk Factors:**
    *   Unexpected changes in business conditions, inflation of raw material costs, labor issues, wars, trade policies, natural disasters, health epidemics, shipping disruptions, port congestions, and cyberattacks.
    *   Supplier insolvency or inability to allocate sufficient production to Tesla.
    *   Production delays or inaccurate demand forecasting.
*   **Mitigation Strategies:**
    *   **Multi-Sourcing:** Where sensible, Tesla works to qualify multiple suppliers for key components to minimize production risks from disruptions.
    *   **Safety Stock:** The company maintains safety stock for key parts and assemblies.
    *   **Die Banks:** Die banks are maintained for components with lengthy procurement lead times.
    *   **Localization:** Localizing and de-risking supply chains across regions, including through vertical integration.

### 6. Impact on Production and Financials

*   **Production Constraints:** Any disruption in the supply of battery cells could limit the production of vehicles and energy storage products. If Tesla cannot meet demand or must procure additional cells at greater costs, it may have to curtail planned production.
*   **Financial Impact:**
    *   **Cost Overruns:** Failure to manage growth effectively may lead to cost overruns.
    *   **Profitability:** Inability to recoup increased raw material costs through price increases may reduce profitability.
    *   **Capital Requirements:** Expanding service and charging capabilities (Supercharger stations) and managing growth requires significant cash investments and management resources.
    *   **Service Capacity:** Delays in adding servicing capacity or issues with vehicle reliability (particularly for high-volume models like Model 3 and Model Y) could overburden servicing capabilities and parts inventory, potentially harming brand reputation and sales.

### Summary Assessment

Tesla’s supply chain strategy is characterized by a dual approach: leveraging global scale for cost efficiencies while aggressively pursuing vertical integration (e.g., in-house battery cells, lithium refining) to reduce dependency on a limited number of critical suppliers (Panasonic, CATL). The primary vulnerabilities lie in the concentration of battery cell sourcing and exposure to volatile raw material markets (lithium, nickel) and geopolitical trade policies (2025 tariffs). Mitigation efforts include multi-sourcing where possible, maintaining safety stocks, and localizing manufacturing in key markets (China, Germany) to insulate against tariff and transportation cost risks. The success of these strategies is contingent on the ability to scale in-house production and manage significant capital investments without incurring prohibitive cost overruns.

## Sources
- Tesla.pdf | Page 15
- Tesla.pdf | Page 32
- Tesla.pdf | Page 27
- Tesla.pdf | Page 25
- Tesla.pdf | Page 33

In [33]:
query = """Evaluate Tesla's competitive positioning and long-term corporate strategy.

Focus on:
- Competition in electric vehicles (traditional OEMs and new entrants)
- Competitive advantages in AI, software, and energy ecosystem
- Diversification into Robotaxi, Optimus Bots, and Energy
- Capital allocation priorities (R&D, capex, buybacks)
- Risks from new competitors and regulatory changes

Explain each point above in very much depth and with intuition


Conclude with overall strategic strengths, weaknesses, and investor implications."""

response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Evaluate Tesla's competitive positioning and long-term corpo...
   Retrieval Quality: 14/17 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 14/17 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7465 | Recall: 0.7679 | F1: 0.7570


# Financial Analysis — Tesla

**Financial Analysis: Tesla’s Competitive Positioning and Long-Term Corporate Strategy**

Based on the retrieved sections from Tesla’s 2025 Annual Report (Form 10-K), the following analysis evaluates the company’s competitive landscape, strategic diversification, and risk profile. Note that specific financial figures for capital allocation (R&D, Capex, Buybacks) and detailed revenue breakdowns are **not disclosed in the retrieved sections**; therefore, the analysis focuses on the qualitative strategic and operational factors provided in the text.

### 1. Competition in Electric Vehicles (Traditional OEMs and New Entrants)

Tesla operates in a "highly competitive" worldwide automotive market that is expected to become "even more competitive in the future." The competitive dynamic is characterized by the following:

*   **Market Saturation and Entry:** A "significant and growing number" of both established and new automobile manufacturers have entered, or announced plans to enter, the electric vehicle (EV) market. This includes competitors offering hybrid, plug-in hybrid, and fully electric vehicles.
*   **Resource Asymmetry:** The context explicitly states that "many of our competitors have significantly more or better-established resources than we do" to devote to design, development, manufacturing, distribution, promotion, sale, and support. These competitors may also achieve "additional cost efficiencies owing to location and economic environments."
*   **Segment-Specific Competition:** Tesla’s vehicles compete based on both traditional segment classification and propulsion technology:
    *   **Cybertruck:** Competes with other pickup trucks.
    *   **Model S and Model X:** Compete primarily with premium sedans and premium SUVs.
    *   **Model 3 and Model Y:** Compete with small to medium-sized sedans and compact SUVs.
    *   All these segments are described as "extremely competitive markets."
*   **Impact of Competition:** Increased competition poses risks of "lower vehicle unit sales, price reductions, revenue shortfalls, loss of customers and loss of market share."

### 2. Competitive Advantages in AI, Software, and Energy Ecosystem

Tesla’s strategic positioning relies on proprietary technology and an integrated ecosystem rather than just vehicle hardware.

*   **Intellectual Property and Innovation:** Tesla places a "strong emphasis on our innovative approach and proprietary designs." The company prioritizes obtaining patents to ensure "freedom to operate" across all products and technologies.
*   **Open-Source Pledge for EVs:** To encourage the advancement of a "common, rapidly-evolving platform for electric vehicles," Tesla has irrevocably pledged not to initiate lawsuits against parties acting in good faith for infringing its patents related to electric vehicles or related equipment. This strategy aims to benefit the broader EV ecosystem, potentially accelerating market adoption that Tesla can then serve.
*   **Energy Ecosystem Integration:** In the energy generation business, Tesla competes based on "price and the ease by which customers can switch." The company cites "aesthetics, superior performance and ease of installation and integration with Powerwall" as key differentiators for its solar panels. This integration suggests a lock-in effect where solar generation and battery storage (Powerwall) are sold as a cohesive system.
*   **AI and Autonomy:** The context highlights that future production growth will be initiated by "advances in autonomy." Tesla is developing "self-driving technology and services" and "vehicle applications and software platforms," positioning AI as a core driver of future value rather than just a feature.

### 3. Diversification into Robotaxi, Optimus Bots, and Energy

Tesla is actively diversifying beyond traditional vehicle sales into high-growth, high-margin potential areas.

*   **Robotaxi Business:** Tesla is launching a "Robotaxi business" and is focused on developing "dedicated infrastructure," including vehicle cleaning, maintenance, charging, security, teleoperations, and fleet management. This indicates a shift from selling cars to potentially selling mobility services.
*   **Optimus Bots:** The company is developing and commercializing "Bots, including Optimus." The context notes this is a "nascent industry that has yet to develop commercially," implying high long-term upside but significant execution risk.
*   **Energy Generation and Storage:** This business is described as having a "significant expansion opportunity." Success is dependent on "incremental volume growth." Tesla competes with traditional local utility companies and other solar energy companies. The company believes the environment is "increasingly conducive to the adoption of renewable energy systems."
*   **Manufacturing Expansion for New Products:** The next phase of production growth is linked to the introduction of new products, including those built on the "next generation vehicle platform" and the ability to "efficiently manufacture our own cells" with "high-volume output, lower capital and production costs and longer range."

### 4. Capital Allocation Priorities (R&D, Capex, Buybacks)

*   **Not Disclosed in the Retrieved Sections:** The provided text does not contain specific figures or explicit statements regarding capital allocation priorities for Research & Development (R&D), Capital Expenditures (Capex), or Share Buybacks.
*   **Implicit Priorities from Context:**
    *   **Manufacturing Capacity:** There is a clear focus on "growing and optimizing our manufacturing capacity," including expanding production at Gigafactories and adding local manufacturing to reduce costs and tariffs.
    *   **Supply Chain Vertical Integration:** Tesla is investing in vertical integration, such as the "in-house lithium refinery in Texas, which began operations in January 2026," to de-risk supply chains and manage raw material costs (aluminum, steel, lithium, nickel, copper).
    *   **Infrastructure:** Significant resources are being directed toward expanding the "Supercharger network" to meet demands from other manufacturers adopting NACS (North American Charging Standard) and for the Robotaxi fleet.

### 5. Risks from New Competitors and Regulatory Changes

*   **Competitive Risks:**
    *   **Resource Disparity:** As noted, competitors may have more resources and better cost efficiencies.
    *   **Price Pressure:** Competition could force "price reductions," directly impacting margins.
    *   **Market Share Erosion:** Loss of customers and market share is a direct risk from increased competition.
*   **Regulatory and Governmental Risks:**
    *   **Incentives:** Government credits and incentives impact customer ownership. However, certain incentives for domestic assembly or local suppliers "may provide a greater benefit to our competitors," potentially negatively impacting Tesla’s profitability.
    *   **Net Metering:** In the solar business, regulators or utilities in certain jurisdictions have "reduced or eliminated the benefit available under net metering." This regulatory change could make Tesla’s energy products less attractive.
    *   **Electricity Prices:** Decreases in retail or wholesale electricity prices from utilities or other renewable sources could make Tesla’s energy products less attractive and lead to "increased rate of customer defaults."
*   **Supply Chain Risks:**
    *   **Raw Material Volatility:** Pricing for aluminum, steel, lithium, nickel, and copper is governed by market conditions and may fluctuate due to supply/demand and speculation.
    *   **Single-Source Dependencies:** Some components are sourced from single suppliers, creating potential production risks if disruptions occur.
*   **Labor Risks:**
    *   **Talent Competition:** There is "strong competition for individuals with skillsets needed for our business," particularly in engineering, AI, and manufacturing. Employees may leave due to a "very competitive labor market" or negative publicity.

### Overall Strategic Assessment

**Strategic Strengths:**
1.  **Integrated Ecosystem:** Tesla’s ability to integrate solar, storage (Powerwall), vehicles, and charging (Supercharger) creates a holistic customer experience that competitors may find difficult to replicate.
2.  **Vertical Integration:** In-house manufacturing of battery cells and raw material processing (lithium refinery) aims to reduce costs and supply chain risks.
3.  **First-Mover in AI/Autonomy:** Focus on autonomy and Robotaxi positions Tesla to capture value from software and services, not just hardware.
4.  **Global Manufacturing Footprint:** Facilities in the U.S., China, and Germany allow for localized production, reducing transportation costs and mitigating tariff impacts.

**Strategic Weaknesses:**
1.  **Resource Asymmetry:** Competitors may have greater financial resources and established cost efficiencies.
2.  **Regulatory Vulnerability:** Dependence on government incentives and net metering policies exposes the business to political and regulatory shifts.
3.  **Execution Risk in New Ventures:** The Optimus Bot and Robotaxi businesses are in "nascent" stages with unproven commercial viability.
4.  **Labor Market Pressure:** High competition for specialized talent in AI and engineering poses a risk to innovation and production capabilities.

**Investor Implications:**
*   **Growth Drivers:** Investors should monitor the success of the Robotaxi launch, the commercialization of Optimus, and volume growth in the energy storage segment.
*   **Margin Pressure:** Increased competition and potential price reductions could pressure gross margins. The ability to maintain cost competitiveness through vertical integration and scale will be critical.
*   **Regulatory Watch:** Changes in net metering policies and government EV incentives will directly impact demand for both vehicles and energy products.
*   **Supply Chain Resilience:** The success of the in-house lithium refinery and battery cell manufacturing will determine Tesla’s ability to control input costs and mitigate supply chain disruptions.

*Note: Specific financial metrics such as Revenue, Operating Income, Net Income, and Capital Allocation figures are not disclosed in the retrieved sections and therefore cannot be included in this analysis.*

## Sources
- Tesla.pdf | Page 21
- Tesla.pdf | Page 33
- Tesla.pdf | Page 15
- Tesla.pdf | Page 28
- Tesla.pdf | Page 20
- Tesla.pdf | Page 55
- Tesla.pdf | Page 57

In [34]:
print(financial_rag._last_context)

---
type: FinancialText
company: Tesla
source_file: Tesla.pdf
page: 21
---

Table of Contents
Energy Generation Systems
The primary competitors to our energy generation business are the traditional local utility companies that supply energy to our
potential customers. We compete with these traditional utility companies primarily based on price and the ease by which customers can
switch to electricity generated by our energy generation systems. We also compete with solar energy companies that provide products
and services similar to ours. Many solar energy companies only install solar energy systems, while others only provide financing for
these installations. We believe we have a significant expansion opportunity with our offerings, including in terms of the aesthetics,
superior performance and ease of installation and integration with Powerwall of our solar panels, and that the environment is
increasingly conducive to the adoption of renewable energy systems. Intellectual Property
We 